# SMPS final five-model verification
Choose **Runtime > Change runtime type > T4 GPU**, then Run all. This notebook embeds the updated local scripts, downloads only official assets, preserves submitted predictions, records failures honestly, and downloads evidence. Existing four-model evidence is never relabelled as five-model evidence.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU runtime'
assert 'T4' in torch.cuda.get_device_name(), 'This benchmark requires a T4'
print(torch.cuda.get_device_name())


In [ ]:
%pip -q install timm==1.0.31 pandas matplotlib scikit-learn opencv-python-headless reportlab pypdf pymupdf
"""Colab asset setup without depending on Git LFS installation."""


import hashlib
import json
import os
from pathlib import Path
import urllib.request
import zipfile

source_revision = os.environ.get('SMPS_SOURCE_REVISION', 'a63df93271c87ca749643a30c4bc188d207f2c31')
project = Path(os.environ.get('SMPS_COLAB_ROOT', '/content/SMPS_FINAL'))
project.mkdir(parents=True, exist_ok=True)
if not (project/'weights/ensemble_config.json').exists():
    archive = Path('/content/smps_source.zip')
    print('[SETUP] Downloading public project source', flush=True)
    urllib.request.urlretrieve('https://codeload.github.com/snehav-Unique/SMPS_MODEL/zip/'+source_revision, archive)
    with zipfile.ZipFile(archive) as bundle:
        for entry in bundle.infolist():
            relative = Path(*Path(entry.filename).parts[1:])
            target = (project/relative).resolve()
            assert target.is_relative_to(project.resolve())
            if not entry.is_dir():
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(bundle.read(entry))
    archive.unlink()
os.chdir(project)
assets = ['weights/efficientnetv2_s_best.pth', 'weights/convnext_tiny_best.pth',
          'weights/swin_tiny_best.pth', 'weights/maxvit_tiny_best.pth',
          'weights/densenet121_best.pth', 'datasets/hackathon_dataset.zip']
expected_assets=json.loads('{"weights/efficientnetv2_s_best.pth": {"sha256": "3e5a3858cd40f22bfb885fd99aaedeff1c049ad720a49755e5e8d15a406d5b6e", "bytes": 81617723}, "weights/convnext_tiny_best.pth": {"sha256": "6165cde373bea423bc3da6e4d24d85f0b13bc44ee0bfcfff9b9f96dedfe18d3e", "bytes": 111355879}, "weights/swin_tiny_best.pth": {"sha256": "c45de824da9aa780f67b3ca597ed7a9352d6b957ce7cd825cb1f3b29019d02f3", "bytes": 110149322}, "weights/maxvit_tiny_best.pth": {"sha256": "10baf3a90f3a717aef75067ab87358c0c07717e827a5c3f2684c76082c56d083", "bytes": 122038277}, "weights/densenet121_best.pth": {"sha256": "683de13e7980c6e1365f877ee58ca81ed43f8e1ed87d0281f2f51f21d19bdc9e", "bytes": 28439894}, "datasets/hackathon_dataset.zip": {"sha256": "3fa8ff24720524a182a4cc449bb56f2ee48a588d04e442a8c3cd93a7089dc08e", "bytes": 133274437}}')
for relative in assets:
    path = project/relative
    text = path.read_text(encoding='utf-8') if path.stat().st_size < 1024 else ''
    if text.startswith('version https://git-lfs.github.com/spec/v1'):
        expected_hash = next(line.split('sha256:')[1] for line in text.splitlines() if line.startswith('oid '))
        expected_size = int(next(line.split()[1] for line in text.splitlines() if line.startswith('size ')))
        partial = path.with_suffix(path.suffix+'.download')
        print('[DOWNLOAD]', relative, f'({expected_size/1048576:.1f} MiB)', flush=True)
        digest = hashlib.sha256()
        with urllib.request.urlopen('https://media.githubusercontent.com/media/snehav-Unique/SMPS_MODEL/'+source_revision+'/'+relative, timeout=120) as response, partial.open('wb') as output:
            while True:
                chunk = response.read(4*1024*1024)
                if not chunk: break
                output.write(chunk)
                digest.update(chunk)
        assert partial.stat().st_size == expected_size, 'Asset size mismatch: '+relative
        assert digest.hexdigest() == expected_hash, 'Asset hash mismatch: '+relative
        partial.replace(path)
        print('[VERIFIED]', relative, flush=True)
    expected=expected_assets[relative]
    assert path.stat().st_size == expected['bytes'], 'Cached asset size mismatch: '+relative
    actual=hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(4*1024*1024),b''):
            actual.update(chunk)
    assert actual.hexdigest() == expected['sha256'], 'Cached asset hash mismatch: '+relative
    print('[VERIFIED ASSET]', relative, flush=True)
print('[SETUP COMPLETE] Official dataset and all five checkpoints are available.', flush=True)

import subprocess


In [ ]:
import json
from pathlib import Path
relocations=json.loads('{"DATASET_AND_WEIGHTS.md": "docs/DATASET_AND_WEIGHTS.md", "Explanatory Video.mp4": "presentation/Explanatory Video.mp4", "FINAL_RUN_README.md": "docs/FINAL_RUN_README.md", "FINAL_STATUS.md": "docs/FINAL_STATUS.md", "README.md": "docs/README_BEFORE_REORGANIZATION.md", "ROBUSTNESS.md": "docs/ROBUSTNESS.md", "SMPS_submission_link.txt": "docs/SMPS_submission_link.txt", "T4_BENCHMARK.md": "docs/T4_BENCHMARK.md", "TEAM_HANDOVER_AND_PROGRESS_REPORT.md": "docs/TEAM_HANDOVER_AND_PROGRESS_REPORT.md", "VIDEO_GUIDE.md": "docs/VIDEO_GUIDE.md", "benchmark_t4.py": "scripts/benchmark_t4.py", "build_maxvit_notebook.py": "scripts/build_maxvit_notebook.py", "build_results.py": "scripts/build_results.py", "calc_correlations.py": "scripts/calc_correlations.py", "check_delivery.py": "scripts/check_delivery.py", "colab_setup.py": "scripts/colab_setup.py", "colab_training.ipynb": "notebooks/training/colab_training.ipynb", "colab_training_convnext.ipynb": "notebooks/training/colab_training_convnext.ipynb", "colab_training_densenet121.ipynb": "notebooks/training/colab_training_densenet121.ipynb", "colab_training_efficientnetv2_s.ipynb": "notebooks/training/colab_training_efficientnetv2_s.ipynb", "colab_training_eva02_tiny.ipynb": "notebooks/training/colab_training_eva02_tiny.ipynb", "colab_training_maxvit_tiny.ipynb": "notebooks/training/colab_training_maxvit_tiny.ipynb", "colab_training_maxvit_tiny_results.ipynb": "notebooks/training/colab_training_maxvit_tiny_results.ipynb", "colab_training_results.ipynb": "notebooks/training/colab_training_results.ipynb", "colab_training_swin_tiny.ipynb": "notebooks/training/colab_training_swin_tiny.ipynb", "colab_training_swin_tiny_results.ipynb": "notebooks/training/colab_training_swin_tiny_results.ipynb", "convnext_results/convnext_results/predictions_convnext.csv": "outputs/models/convnext_tiny/predictions_convnext.csv", "convnext_results/convnext_results/train_log_convnext.csv": "outputs/models/convnext_tiny/train_log_convnext.csv", "convnext_results/convnext_results/training_curves_convnext.png": "outputs/models/convnext_tiny/training_curves_convnext.png", "create_ensemble_bundle.py": "scripts/create_ensemble_bundle.py", "create_split.py": "scripts/create_split.py", "densenet121_results/densenet121_results/predictions_densenet121.csv": "outputs/models/densenet121/predictions_densenet121.csv", "densenet121_results/densenet121_results/train_log_densenet121.csv": "outputs/models/densenet121/train_log_densenet121.csv", "eda.py": "scripts/eda.py", "efficientnetv2_s_results/efficientnetv2_s_results/predictions_efficientnetv2_s.csv": "outputs/models/efficientnetv2_s/predictions_efficientnetv2_s.csv", "efficientnetv2_s_results/efficientnetv2_s_results/train_log_efficientnetv2_s.csv": "outputs/models/efficientnetv2_s/train_log_efficientnetv2_s.csv", "ensemble_blend.py": "scripts/ensemble_blend.py", "eva02_tiny_results/eva02_tiny_results/eva02_tiny_best.pth": "outputs/models/eva02_tiny/eva02_tiny_best.pth", "eva02_tiny_results/eva02_tiny_results/predictions_eva02_tiny.csv": "outputs/models/eva02_tiny/predictions_eva02_tiny.csv", "eva02_tiny_results/eva02_tiny_results/train_log_eva02_tiny.csv": "outputs/models/eva02_tiny/train_log_eva02_tiny.csv", "generate_2page_report.py": "scripts/generate_2page_report.py", "generate_report_pdf.py": "scripts/generate_report_pdf.py", "notebooks/colab_training_maxvit_tiny.ipynb": "notebooks/training/colab_training_maxvit_tiny_template.ipynb", "predictions_convnext.csv": "outputs/predictions/predictions_convnext.csv", "predictions_ensemble.csv": "outputs/predictions/predictions_ensemble.csv", "predictions_maxvit.csv": "outputs/predictions/predictions_maxvit.csv", "predictions_original.csv": "outputs/predictions/predictions_original.csv", "prepare_final_delivery.py": "scripts/prepare_final_delivery.py", "prepare_t4_notebook.py": "scripts/prepare_t4_notebook.py", "render_dashboard.py": "scripts/render_dashboard.py", "report-sneha.pdf": "docs/reports/report-sneha.pdf", "report.pdf": "docs/reports/report.pdf", "report_detailed.pdf": "docs/reports/report_detailed.pdf", "requirements.txt": "docs/environments/requirements_original_colab.txt", "results.html": "presentation/results.html", "robustness_low_memory.py": "scripts/robustness_low_memory.py", "robustness_model_summary.py": "scripts/robustness_model_summary.py", "robustness_test.py": "scripts/robustness_test.py", "robustness_verify.py": "scripts/robustness_verify.py", "swin_results/swin_results/predictions_swin.csv": "outputs/models/swin_tiny/predictions_swin.csv", "swin_results/swin_results/train_log_swin.csv": "outputs/models/swin_tiny/train_log_swin.csv", "validate_final.py": "scripts/validate_final.py", "verify_final.py": "scripts/verify_final.py", "verify_t4_results.py": "scripts/verify_t4_results.py"}')
for old,new in relocations.items():
    src,dst=Path(old),Path(new)
    if src.is_file() and not dst.exists():
        dst.parent.mkdir(parents=True,exist_ok=True)
        src.rename(dst)
scripts=json.loads('{"inference_script.py": "\\"\\"\\"\\ninference_script.py\\n===================\\nOfficial inference script for the Deepfake ML Hackathon (IIIT Bangalore).\\nFilled from inference_template.py by Member 1.\\n\\nUsage (as required by the hackathon):\\n    python inference_script.py \\\\\\n        --test_dir ./hackathon_dataset/test/images/ \\\\\\n        --weights  ./weights/best_model.pth \\\\\\n        --output   predictions.csv\\n\\nOptional flags:\\n    --backbone     Model backbone name (default: efficientnet_b4)\\n    --batch_size   Inference batch size (default: 64)\\n    --threshold    Fake probability threshold (default: 0.5)\\n    --fp16         Use FP16 inference (faster on T4, default: enabled if CUDA)\\n    --tta          Use test-time augmentation (3x slower but potentially better AUC)\\n\\nAll paths are via CLI arguments - no hardcoded paths (required by hackathon rules).\\n\\"\\"\\"\\n\\nimport os\\nimport sys\\nimport argparse\\nimport json\\nimport time\\nif __name__ == \'__main__\' and \'--ensemble_worker\' not in sys.argv:\\n    from ensemble_worker_control import run_if_ensemble\\n    ensemble_exit = run_if_ensemble(sys.argv[1:])\\n    if ensemble_exit is not None:\\n        sys.exit(ensemble_exit)\\nimport numpy as np\\nimport pandas as pd\\nfrom pathlib import Path\\n\\nimport torch\\nimport torch.nn as nn\\nfrom torch.utils.data import Dataset, DataLoader\\nfrom torchvision import transforms\\nfrom PIL import Image\\n\\n\\n# -----------------------------------------------------------------------------\\n# CONSTANTS (overrideable via CLI)\\n# -----------------------------------------------------------------------------\\nIMAGE_SIZE   = 224\\nBATCH_SIZE   = 64\\n# Windows: num_workers>0 with DataLoader requires spawn context - safe inside\\n# __main__ guard. Set to 0 on CPU to avoid overhead; GPU gets full workers.\\nNUM_WORKERS  = 4 if torch.cuda.is_available() else 0\\nTHRESHOLD    = 0.5\\nDEVICE       = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\nBACKBONE     = \\"efficientnet_b4\\"\\nPROGRESS_PATH = None\\nRAW_OUTPUT = False\\n\\n\\ndef write_progress(**values):\\n    if PROGRESS_PATH:\\n        path = Path(PROGRESS_PATH)\\n        path.parent.mkdir(parents=True, exist_ok=True)\\n        temp = path.with_suffix(\'.tmp\')\\n        temp.write_text(json.dumps(values), encoding=\'utf-8\')\\n        temp.replace(path)\\n\\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\\nIMAGENET_STD  = [0.229, 0.224, 0.225]\\n\\n\\n# -----------------------------------------------------------------------------\\n# DATASET - Flat directory of test images, no labels\\n# -----------------------------------------------------------------------------\\nclass TestDataset(Dataset):\\n    def __init__(self, image_dir: str, transform=None):\\n        self.image_dir = Path(image_dir)\\n        self.image_paths = sorted(\\n            list(self.image_dir.glob(\\"*.jpg\\"))\\n            + list(self.image_dir.glob(\\"*.jpeg\\"))\\n            + list(self.image_dir.glob(\\"*.png\\"))\\n        )\\n        self.transform = transform\\n\\n        if len(self.image_paths) == 0:\\n            raise FileNotFoundError(f\\"No images found in: {self.image_dir}\\")\\n        print(f\\"[INFO] Found {len(self.image_paths)} test images in {self.image_dir}\\")\\n\\n    def __len__(self):\\n        return len(self.image_paths)\\n\\n    def __getitem__(self, idx):\\n        img_path = self.image_paths[idx]\\n        image_id = img_path.stem   # UUID without .jpg extension\\n\\n        try:\\n            img = Image.open(img_path).convert(\\"RGB\\")\\n        except Exception as e:\\n            print(f\\"[WARNING] Could not read {img_path}: {e}. Using blank image.\\")\\n            raise RuntimeError(f\\"Unreadable test image: {img_path}\\") from e\\n\\n        if self.transform:\\n            img = self.transform(img)\\n\\n        return img, image_id\\n\\n\\n# -----------------------------------------------------------------------------\\n# MODEL - Architecture must match what was trained\\n# -----------------------------------------------------------------------------\\ndef build_model(backbone: str) -> nn.Module:\\n    \\"\\"\\"Build the model architecture (without loading weights).\\"\\"\\"\\n\\n    if backbone == \\"efficientnet_b4\\":\\n        from torchvision.models import efficientnet_b4\\n        model = efficientnet_b4(weights=None)\\n        in_features = model.classifier[1].in_features\\n        model.classifier = nn.Sequential(\\n            nn.Dropout(p=0.4),\\n            nn.Linear(in_features, 1),\\n        )\\n\\n    elif backbone == \\"efficientnet_b2\\":\\n        from torchvision.models import efficientnet_b2\\n        model = efficientnet_b2(weights=None)\\n        in_features = model.classifier[1].in_features\\n        model.classifier = nn.Sequential(\\n            nn.Dropout(p=0.3),\\n            nn.Linear(in_features, 1),\\n        )\\n\\n    elif backbone == \\"convnext_tiny\\":\\n        try:\\n            import timm\\n            model = timm.create_model(\\"convnext_tiny\\", pretrained=False, num_classes=1)\\n        except ImportError:\\n            raise ImportError(\\"timm is required for convnext_tiny. Run: pip install timm\\")\\n\\n    elif backbone in (\\"efficientnetv2_s\\", \\"tf_efficientnetv2_s\\"):\\n        try:\\n            import timm\\n            model = timm.create_model(\\"tf_efficientnetv2_s\\", pretrained=False, num_classes=1)\\n        except ImportError:\\n            raise ImportError(\\"timm is required. Run: pip install timm\\")\\n\\n    elif backbone in (\\"swin_tiny\\", \\"swin_tiny_patch4_window7_224\\"):\\n        try:\\n            import timm\\n            model = timm.create_model(\\"swin_tiny_patch4_window7_224\\", pretrained=False, num_classes=1)\\n        except ImportError:\\n            raise ImportError(\\"timm is required. Run: pip install timm\\")\\n\\n    elif backbone == \\"densenet121\\":\\n        try:\\n            import timm\\n            model = timm.create_model(\\"densenet121\\", pretrained=False, num_classes=1)\\n        except ImportError:\\n            raise ImportError(\\"timm is required. Run: pip install timm\\")\\n\\n    elif backbone in (\\"maxvit_tiny\\", \\"maxvit_tiny_tf_224\\"):\\n        try:\\n            import timm\\n            model = timm.create_model(\\"maxvit_tiny_tf_224\\", pretrained=False, num_classes=1)\\n        except ImportError:\\n            raise ImportError(\\"timm is required. Run: pip install timm\\")\\n\\n    else:\\n        raise ValueError(f\\"Unknown backbone: {backbone!r}\\")\\n\\n    return model\\n\\n\\nclass EnsembleRunner:\\n    \\"\\"\\"Wrapper that runs the multi-model ensemble with horizontal-flip TTA.\\"\\"\\"\\n    def __init__(self, models_and_weights, device):\\n        self.models_and_weights = models_and_weights\\n        self.device = device\\n\\n    def to(self, device):\\n        for m, _ in self.models_and_weights:\\n            m.to(device)\\n        self.device = device\\n        return self\\n\\n    def eval(self):\\n        for m, _ in self.models_and_weights:\\n            m.eval()\\n        return self\\n\\n    def half(self):\\n        for m, _ in self.models_and_weights:\\n            m.half()\\n        return self\\n\\n    def __call__(self, x):\\n        total = None\\n        for model, weight in self.models_and_weights:\\n            p = (\\n                torch.sigmoid(model(x)).squeeze(-1) +\\n                torch.sigmoid(model(x.flip(-1))).squeeze(-1)\\n            ) / 2.0\\n            if total is None:\\n                total = weight * p\\n            else:\\n                total = total + weight * p\\n        return total\\n\\n\\nclass SequentialEnsemble:\\n    def __init__(self, config, base_dir):\\n        self.config = config\\n        self.base_dir = base_dir\\n\\n\\ndef load_model(weights_path: str, backbone: str, device: str, sequential=False):\\n    \\"\\"\\"Load model and weights onto device (supports single model, JSON config, or ensemble bundle).\\"\\"\\"\\n    import json\\n    w_path = Path(weights_path)\\n\\n    # Check if directory or JSON config\\n    if w_path.is_dir():\\n        config_file = w_path / \\"ensemble_config.json\\"\\n        if config_file.exists():\\n            w_path = config_file\\n        else:\\n            pths = sorted(list(w_path.glob(\\"*.pth\\")))\\n            if pths:\\n                w_path = pths[0]\\n            else:\\n                raise FileNotFoundError(f\\"No model checkpoint or ensemble_config.json found in {weights_path}\\")\\n\\n    if w_path.suffix.lower() == \\".json\\":\\n        with open(w_path, \\"r\\", encoding=\\"utf-8\\") as f:\\n            cfg = json.load(f)\\n        if sequential:\\n            return SequentialEnsemble(cfg, w_path.parent)\\n        print(f\\"[INFO] Loading Super-Ensemble ({len(cfg[\'models\'])} models) via {w_path.name} on {device} ...\\")\\n        models_and_weights = []\\n        base_dir = w_path.parent\\n        for m_info in cfg[\\"models\\"]:\\n            m = build_model(m_info[\\"arch\\"])\\n            m_pth = base_dir / m_info[\\"file\\"]\\n            state = torch.load(m_pth, map_location=device)\\n            if isinstance(state, dict) and \\"state_dict\\" in state:\\n                state = state[\\"state_dict\\"]\\n            elif isinstance(state, dict) and \\"model\\" in state:\\n                state = state[\\"model\\"]\\n            m.load_state_dict(state)\\n            m.to(device)\\n            m.eval()\\n            models_and_weights.append((m, m_info[\\"weight\\"]))\\n            print(f\\"  Loaded {m_info[\'name\']} ({m_info[\'arch\']}) with weight {m_info[\'weight\']}\\")\\n        return EnsembleRunner(models_and_weights, device)\\n\\n    state = torch.load(w_path, map_location=device)\\n\\n    if isinstance(state, dict) and state.get(\\"is_ensemble\\"):\\n        print(f\\"[INFO] Detected unified ensemble bundle with {len(state[\'models\'])} models from {weights_path}\\")\\n        models_and_weights = []\\n        for (name, arch), weight in zip(state[\\"models\\"], state[\\"weights\\"]):\\n            m = build_model(arch)\\n            s = state[\\"state_dicts\\"][name]\\n            m.load_state_dict(s)\\n            m.to(device)\\n            m.eval()\\n            models_and_weights.append((m, weight))\\n            print(f\\"  Loaded ensemble model {name} ({arch}) with weight {weight}\\")\\n        return EnsembleRunner(models_and_weights, device)\\n\\n    model = build_model(backbone)\\n    if isinstance(state, dict) and \\"state_dict\\" in state:\\n        model.load_state_dict(state[\\"state_dict\\"])\\n    elif isinstance(state, dict) and \\"model\\" in state:\\n        model.load_state_dict(state[\\"model\\"])\\n    elif isinstance(state, dict):\\n        model.load_state_dict(state)\\n    else:\\n        model = state  # Full model saved\\n\\n    model.to(device)\\n    model.eval()\\n    print(f\\"[INFO] Model ({backbone}) loaded from {weights_path} on {device}\\")\\n    return model\\n\\n\\n# -----------------------------------------------------------------------------\\n# TRANSFORMS - MUST match val/inference transforms used during training\\n# -----------------------------------------------------------------------------\\ndef get_inference_transforms():\\n    \\"\\"\\"Standard inference transforms - matches val transforms in augmentations.py.\\"\\"\\"\\n    return transforms.Compose([\\n        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\\n        transforms.ToTensor(),\\n        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),\\n    ])\\n\\n\\ndef get_tta_transforms():\\n    \\"\\"\\"\\n    Three TTA variants: original, horizontal flip, brightness shift.\\n    If --tta is set, inference averages predictions from all three.\\n    \\"\\"\\"\\n    return [\\n        transforms.Compose([\\n            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\\n            transforms.ToTensor(),\\n            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\\n        ]),\\n        transforms.Compose([\\n            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\\n            transforms.RandomHorizontalFlip(p=1.0),\\n            transforms.ToTensor(),\\n            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\\n        ]),\\n        transforms.Compose([\\n            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),\\n            transforms.ColorJitter(brightness=0.1),\\n            transforms.ToTensor(),\\n            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),\\n        ]),\\n    ]\\n\\n\\n# -----------------------------------------------------------------------------\\n# INFERENCE LOOP\\n# -----------------------------------------------------------------------------\\n@torch.no_grad()\\ndef run_inference_single(model, test_dir: str, transform, batch_size: int,\\n                         device: str, use_fp16: bool) -> pd.DataFrame:\\n    \\"\\"\\"Run inference with a single transform (standard or one TTA variant).\\"\\"\\"\\n    dataset = TestDataset(test_dir, transform=transform)\\n    loader  = DataLoader(\\n        dataset,\\n        batch_size=batch_size,\\n        shuffle=False,\\n        num_workers=NUM_WORKERS,\\n        pin_memory=(device == \\"cuda\\"),\\n    )\\n\\n    all_ids, all_probs = [], []\\n    total_batches = len(loader)\\n\\n    for i, (batch_imgs, batch_ids) in enumerate(loader):\\n        batch_imgs = batch_imgs.to(device)\\n\\n        if use_fp16 and device == \\"cuda\\":\\n            batch_imgs = batch_imgs.half()\\n\\n        if isinstance(model, EnsembleRunner):\\n            probs = model(batch_imgs).float().cpu().numpy()\\n        else:\\n            logits = model(batch_imgs)              # (B, 1) or (B,)\\n            probs  = torch.sigmoid(logits).squeeze(-1).float().cpu().numpy()\\n\\n        all_ids.extend(list(batch_ids))\\n        all_probs.extend(probs.tolist())\\n\\n        write_progress(stage=\'inference\', processed=len(all_ids), total=len(dataset),\\n                       batch=i + 1, batches=total_batches, device=device)\\n        if (i + 1) % 10 == 0 or (i + 1) == total_batches:\\n            print(f\\"  Batch [{i+1:4d}/{total_batches}] | \\"\\n                  f\\"Processed {len(all_ids)} images\\", flush=True)\\n\\n    print()\\n    return pd.DataFrame({\\"image_id\\": all_ids, \\"confidence\\": all_probs})\\n\\n\\ndef run_inference(model, test_dir: str, batch_size: int, device: str,\\n                  use_fp16: bool, use_tta: bool, threshold: float) -> pd.DataFrame:\\n    \\"\\"\\"\\n    Main inference runner. Supports standard and TTA inference.\\n    Returns DataFrame with columns: image_id, label, confidence\\n    \\"\\"\\"\\n    if isinstance(model, SequentialEnsemble):\\n        import gc\\n        accumulator = None\\n        ids = None\\n        for info in model.config[\'models\']:\\n            print(f\\"[MODEL] {info[\'name\']} | blend weight {info[\'weight\']}\\", flush=True)\\n            single = load_model(str(model.base_dir/info[\'file\']), info[\'arch\'], device)\\n            wrapped = EnsembleRunner([(single, 1.0)], device)\\n            if use_fp16 and device == \'cuda\':\\n                wrapped.half()\\n            passes = get_tta_transforms() if use_tta else [get_inference_transforms()]\\n            predictions = [run_inference_single(wrapped, test_dir, tf, batch_size, device, use_fp16) for tf in passes]\\n            current = np.mean([p.confidence.to_numpy(dtype=np.float32) for p in predictions], axis=0)\\n            if ids is None:\\n                ids = predictions[0].image_id.to_list()\\n                accumulator = np.zeros(len(ids), dtype=np.float32)\\n            assert ids == predictions[0].image_id.to_list()\\n            accumulator += info[\'weight\'] * current\\n            del single, wrapped, predictions\\n            gc.collect()\\n            if device == \'cuda\':\\n                torch.cuda.empty_cache()\\n        result = pd.DataFrame({\'image_id\': ids, \'confidence\': accumulator})\\n        result[\'label\'] = (result.confidence >= threshold).astype(int)\\n        result[\'confidence\'] = result.confidence.round(6)\\n        return result[[\'image_id\', \'label\', \'confidence\']]\\n\\n    if use_fp16 and device == \\"cuda\\":\\n        model = model.half()\\n        print(\\"[INFO] FP16 inference enabled.\\")\\n\\n    if use_tta:\\n        print(f\\"[INFO] Running TTA inference (3 augmentations)...\\")\\n        tta_transforms = get_tta_transforms()\\n        all_dfs = []\\n        for idx, tf in enumerate(tta_transforms):\\n            print(f\\"  TTA pass {idx+1}/3...\\")\\n            df = run_inference_single(model, test_dir, tf, batch_size, device, use_fp16)\\n            all_dfs.append(df.set_index(\\"image_id\\")[\\"confidence\\"])\\n\\n        # Average all TTA predictions\\n        avg_probs = pd.concat(all_dfs, axis=1).mean(axis=1).reset_index()\\n        avg_probs.columns = [\\"image_id\\", \\"confidence\\"]\\n        result = avg_probs\\n    else:\\n        print(f\\"[INFO] Running standard inference...\\")\\n        result = run_inference_single(\\n            model, test_dir,\\n            get_inference_transforms(),\\n            batch_size, device, use_fp16\\n        )\\n\\n    result[\\"label\\"]      = (result[\\"confidence\\"] >= threshold).astype(int)\\n    if not RAW_OUTPUT:\\n        result[\\"confidence\\"] = result[\\"confidence\\"].round(6)\\n    result = result[[\\"image_id\\", \\"label\\", \\"confidence\\"]]  # Required column order\\n\\n    return result\\n\\n\\n# -----------------------------------------------------------------------------\\n# VALIDATION\\n# -----------------------------------------------------------------------------\\ndef validate_output(df: pd.DataFrame, expected_count: int = None):\\n    \\"\\"\\"Validate that predictions.csv meets hackathon requirements.\\"\\"\\"\\n    errors = []\\n    if \\"image_id\\"   not in df.columns: errors.append(\\"Missing column: image_id\\")\\n    if \\"label\\"      not in df.columns: errors.append(\\"Missing column: label\\")\\n    if \\"confidence\\" not in df.columns: errors.append(\\"Missing column: confidence\\")\\n\\n    if not errors:\\n        if df.isnull().any().any():\\n            errors.append(\\"Missing values in predictions\\")\\n        if not df[\\"confidence\\"].between(0, 1).all():\\n            errors.append(\\"Confidence values outside [0, 1]\\")\\n        if not df[\\"label\\"].isin([0, 1]).all():\\n            errors.append(\\"Labels must be 0 or 1 only\\")\\n        if df[\\"image_id\\"].duplicated().any():\\n            errors.append(f\\"Duplicate image_ids: {df[\'image_id\'].duplicated().sum()}\\")\\n        if expected_count and len(df) != expected_count:\\n            errors.append(f\\"Row count mismatch: got {len(df)}, expected {expected_count}\\")\\n\\n    if errors:\\n        for e in errors:\\n            print(f\\"[ERROR] {e}\\")\\n        raise ValueError(\\"Output validation failed. See errors above.\\")\\n\\n\\n# -----------------------------------------------------------------------------\\n# MAIN\\n# -----------------------------------------------------------------------------\\ndef main():\\n    global PROGRESS_PATH, RAW_OUTPUT\\n    started = time.perf_counter()\\n    parser = argparse.ArgumentParser(\\n        description=\\"Deepfake ML Hackathon - Inference Script (IIIT Bangalore)\\"\\n    )\\n    parser.add_argument(\\"--test_dir\\",   required=True,\\n                        help=\\"Path to test images directory (flat, no subfolders)\\")\\n    parser.add_argument(\\"--weights\\",    required=True,\\n                        help=\\"Path to model weights file (.pth)\\")\\n    parser.add_argument(\\"--output\\",     default=\\"predictions.csv\\",\\n                        help=\\"Output CSV path (default: predictions.csv)\\")\\n    parser.add_argument(\\"--backbone\\",   default=BACKBONE,\\n                        help=f\\"Model backbone (default: {BACKBONE})\\")\\n    parser.add_argument(\\"--batch_size\\", type=int,   default=BATCH_SIZE,\\n                        help=f\\"Inference batch size (default: {BATCH_SIZE})\\")\\n    parser.add_argument(\\"--threshold\\",  type=float, default=THRESHOLD,\\n                        help=f\\"Fake probability threshold (default: {THRESHOLD})\\")\\n    parser.add_argument(\\"--fp16\\",       action=\\"store_true\\", default=False,\\n                        help=\\"Use FP16 inference on CUDA (default: True)\\")\\n    parser.add_argument(\\"--tta\\",        action=\\"store_true\\", default=False,\\n                        help=\\"Use test-time augmentation (slower, may improve AUC)\\")\\n    parser.add_argument(\'--progress_json\', help=\'Optional live dashboard progress file\')\\n    parser.add_argument(\'--metrics_json\', help=\'Optional runtime and GPU memory evidence\')\\n    parser.add_argument(\'--threads\', type=int, default=4, help=\'CPU intra-op threads\')\\n    parser.add_argument(\'--ensemble_worker\', action=\'store_true\', help=argparse.SUPPRESS)\\n    parser.add_argument(\'--flip_tta\', action=\'store_true\', help=argparse.SUPPRESS)\\n    args = parser.parse_args()\\n    PROGRESS_PATH = args.progress_json\\n    RAW_OUTPUT = args.ensemble_worker\\n    if args.threads < 1:\\n        parser.error(\'--threads must be positive\')\\n    torch.set_num_threads(args.threads)\\n    write_progress(stage=\'loading\', processed=0, total=3000)\\n    if torch.cuda.is_available():\\n        torch.cuda.reset_peak_memory_stats()\\n\\n    device = DEVICE\\n    print(f\\"\\\\n{\'=\'*60}\\")\\n    print(f\\"  Deepfake ML Hackathon - Inference\\")\\n    print(f\\"  Test dir  : {args.test_dir}\\")\\n    print(f\\"  Weights   : {args.weights}\\")\\n    print(f\\"  Output    : {args.output}\\")\\n    print(f\\"  Backbone  : {args.backbone}\\")\\n    print(f\\"  Device    : {device}\\")\\n    print(f\\"  Batch     : {args.batch_size}\\")\\n    print(f\\"  Threshold : {args.threshold}\\")\\n    print(f\\"  FP16      : {args.fp16 and device == \'cuda\'}\\")\\n    print(f\\"  TTA       : {args.tta}\\")\\n    print(f\\"{\'=\'*60}\\\\n\\")\\n\\n    # -- Load model --------------------------------------------------------\\n    model = load_model(args.weights, args.backbone, device, sequential=True)\\n    if args.flip_tta:\\n        model = EnsembleRunner([(model, 1.0)], device)\\n\\n    # -- Run inference -----------------------------------------------------\\n    predictions = run_inference(\\n        model,\\n        test_dir=args.test_dir,\\n        batch_size=args.batch_size,\\n        device=device,\\n        use_fp16=args.fp16,\\n        use_tta=args.tta,\\n        threshold=args.threshold,\\n    )\\n\\n    # -- Validate ----------------------------------------------------------\\n    validate_output(predictions, expected_count=3000)\\n\\n    # -- Save --------------------------------------------------------------\\n    predictions.to_csv(args.output, index=False)\\n    if device == \'cuda\':\\n        torch.cuda.synchronize()\\n    metrics = dict(stage=\'complete\', processed=len(predictions), total=len(predictions),\\n                   device=device, seconds=time.perf_counter() - started,\\n                   device_name=torch.cuda.get_device_name() if device == \'cuda\' else \'CPU\',\\n                   peak_allocated_mib=torch.cuda.max_memory_allocated()/1048576 if device == \'cuda\' else None)\\n    write_progress(**metrics)\\n    if args.metrics_json:\\n        Path(args.metrics_json).write_text(json.dumps(metrics, indent=2), encoding=\'utf-8\')\\n\\n    print(f\\"\\\\n[DONE] Predictions saved to: {args.output}\\")\\n    print(f\\"  Total images   : {len(predictions)}\\")\\n    print(f\\"  Predicted FAKE : {predictions[\'label\'].sum()} ({predictions[\'label\'].mean()*100:.1f}%)\\")\\n    print(f\\"  Predicted REAL : {(predictions[\'label\']==0).sum()}\\")\\n    print(f\\"  Mean confidence: {predictions[\'confidence\'].mean():.4f}\\")\\n    print(f\\"  Min confidence : {predictions[\'confidence\'].min():.4f}\\")\\n    print(f\\"  Max confidence : {predictions[\'confidence\'].max():.4f}\\")\\n\\n\\nif __name__ == \\"__main__\\":\\n    main()\\n", "ensemble_worker_control.py": "\\"\\"\\"Stdlib-only ensemble controller: isolate model memory in short-lived workers.\\"\\"\\"\\nimport argparse\\nimport csv\\nimport json\\nimport math\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\nimport tempfile\\nimport time\\n\\n\\ndef run_if_ensemble(argv):\\n    parser=argparse.ArgumentParser(add_help=False)\\n    parser.add_argument(\'--weights\')\\n    known,_=parser.parse_known_args(argv)\\n    if not known.weights:return None\\n    config_path=Path(known.weights)\\n    if config_path.is_dir():config_path=config_path/\'ensemble_config.json\'\\n    if config_path.suffix.lower()!=\'.json\':return None\\n    parser.add_argument(\'--test_dir\',required=True)\\n    parser.add_argument(\'--output\',default=\'predictions.csv\')\\n    parser.add_argument(\'--batch_size\',type=int,default=64)\\n    parser.add_argument(\'--threads\',type=int,default=4)\\n    parser.add_argument(\'--threshold\',type=float,default=.5)\\n    parser.add_argument(\'--progress_json\')\\n    parser.add_argument(\'--metrics_json\')\\n    parser.add_argument(\'--fp16\',action=\'store_true\')\\n    parser.add_argument(\'--tta\',action=\'store_true\')\\n    parser.add_argument(\'--backbone\')\\n    args=parser.parse_args(argv)\\n    if args.tta:parser.error(\'JSON ensemble already uses deterministic original+horizontal-flip TTA; omit --tta.\')\\n    started=time.perf_counter()\\n    config=json.loads(config_path.read_text())\\n    assert abs(sum(m[\'weight\'] for m in config[\'models\'])-1)<1e-8\\n    assert all(m[\'weight\']>=0 for m in config[\'models\'])\\n    output=Path(args.output).resolve();output.parent.mkdir(parents=True,exist_ok=True)\\n    combined={};order=None;measurements=[]\\n    # Use output drive for scratch files; avoids the nearly full system TEMP drive.\\n    with tempfile.TemporaryDirectory(prefix=\'smps_workers_\',dir=output.parent) as scratch:\\n      for index,info in enumerate(config[\'models\']):\\n        target=Path(scratch)/(info[\'name\']+\'.csv\');metrics=target.with_suffix(\'.json\')\\n        print(f\'\\\\n[ENSEMBLE {index+1}/{len(config[\\"models\\"])}] {info[\\"name\\"]} | weight={info[\\"weight\\"]}\',flush=True)\\n        command=[sys.executable,\'-u\',str(Path(__file__).with_name(\'inference_script.py\')),\\n                 \'--ensemble_worker\',\'--flip_tta\',\'--test_dir\',args.test_dir,\\n                 \'--weights\',str(config_path.parent/info[\'file\']),\'--backbone\',info[\'arch\'],\\n                 \'--output\',str(target),\'--batch_size\',str(args.batch_size),\'--threads\',str(args.threads),\\n                 \'--metrics_json\',str(metrics)]\\n        if args.progress_json:command+=[\'--progress_json\',args.progress_json]\\n        subprocess.run(command,check=True)\\n        measurements.append(json.loads(metrics.read_text()))\\n        with target.open(newline=\'\') as stream:rows=list(csv.DictReader(stream))\\n        ids=[r[\'image_id\'] for r in rows]\\n        assert len(ids)==3000 and len(set(ids))==3000\\n        if order is None:\\n            order=ids;combined=dict.fromkeys(ids,0.)\\n        assert ids==order\\n        for row in rows:\\n            value=float(row[\'confidence\']);assert math.isfinite(value) and 0<=value<=1\\n            combined[row[\'image_id\']]+=info[\'weight\']*value\\n    with output.open(\'w\',newline=\'\') as stream:\\n        writer=csv.writer(stream);writer.writerow([\'image_id\',\'label\',\'confidence\'])\\n        for key in order:\\n            value=combined[key]\\n            writer.writerow([key,int(value>=args.threshold),f\'{value:.6f}\'])\\n    result=dict(stage=\'complete\',processed=3000,total=3000,models=len(config[\'models\']),\\n                device=measurements[0][\'device\'],device_name=measurements[0][\'device_name\'],\\n                seconds=time.perf_counter()-started,\\n                peak_allocated_mib=max((m[\'peak_allocated_mib\'] or 0) for m in measurements) or None,\\n                per_model=measurements,execution=\'isolated sequential workers; original+horizontal_flip\')\\n    if args.metrics_json:Path(args.metrics_json).write_text(json.dumps(result,indent=2))\\n    if args.progress_json:Path(args.progress_json).write_text(json.dumps(result))\\n    print(f\'[ENSEMBLE COMPLETE] {len(order)} images; {result[\\"seconds\\"]:.2f} seconds; {result[\\"device_name\\"]}\',flush=True)\\n    return 0\\n", "scripts/verify_final.py": "\\"\\"\\"Verify saved blending, then optionally reproduce predictions from real weights.\\n\\nNever overwrites predictions.csv. CPU timing is never presented as T4 timing.\\n\\"\\"\\"\\n\\nimport sys as _sys\\nfrom pathlib import Path as _Path\\n_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))\\n\\nimport argparse\\nimport hashlib\\nimport json\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\nimport time\\nimport numpy as np\\nimport pandas as pd\\n\\nROOT = Path(__file__).resolve().parent.parent\\nOUT = ROOT / \'outputs/reproduction/verification\'\\n\\n\\ndef compare(reference, actual, tolerance):\\n    for frame in (reference, actual):\\n        assert list(frame.columns) == [\'image_id\', \'label\', \'confidence\']\\n        assert len(frame) == 3000 and frame.image_id.nunique() == 3000\\n        assert not frame.isna().any().any()\\n        assert frame.confidence.between(0, 1).all() and frame.label.isin([0, 1]).all()\\n    assert set(reference.image_id) == set(actual.image_id), \'Image IDs differ\'\\n    a = reference.set_index(\'image_id\').sort_index()\\n    b = actual.set_index(\'image_id\').sort_index()\\n    delta = np.abs(a.confidence.to_numpy() - b.confidence.to_numpy())\\n    labels = int((a.label != b.label).sum())\\n    return dict(passed=bool(delta.max() <= tolerance and labels == 0),\\n                max_probability_difference=float(delta.max()),\\n                mean_probability_difference=float(delta.mean()),\\n                label_mismatches=labels, tolerance=tolerance, images=3000)\\n\\n\\ndef main():\\n    global OUT\\n    parser = argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--test-dir\', type=Path)\\n    parser.add_argument(\'--output-dir\', type=Path, default=OUT)\\n    parser.add_argument(\'--runs\', type=int, default=1)\\n    parser.add_argument(\'--batch-size\', type=int, default=64)\\n    parser.add_argument(\'--tolerance\', type=float, default=1e-5)\\n    parser.add_argument(\'--weights-dir\', type=Path, default=ROOT/\'weights\')\\n    parser.add_argument(\'--threads\', type=int, default=4)\\n    args = parser.parse_args()\\n    OUT = args.output_dir.resolve()\\n    if OUT == (ROOT/\'outputs/final_verification\').resolve():\\n        parser.error(\'Recorded evidence is immutable; choose a new output directory.\')\\n    if (OUT/\'verification.json\').exists():\\n        parser.error(\'Output already contains a run; choose a new --output-dir.\')\\n    assert args.runs > 0 and args.batch_size > 0 and args.tolerance >= 0\\n    OUT.mkdir(parents=True, exist_ok=True)\\n    weights_dir=args.weights_dir.resolve()\\n    cfg = json.loads((weights_dir/\'ensemble_config.json\').read_text(encoding=\'utf-8\'))\\n    sources = {\\n        \'efficientnetv2_s\': \'outputs/models/efficientnetv2_s/predictions_efficientnetv2_s.csv\',\\n        \'convnext_tiny\': \'outputs/predictions/predictions_convnext.csv\',\\n        \'swin_tiny\': \'outputs/models/swin_tiny/predictions_swin.csv\',\\n        \'maxvit_tiny\': \'outputs/predictions/predictions_maxvit.csv\',\\n        \'densenet121\': \'outputs/models/densenet121/predictions_densenet121.csv\',\\n    }\\n    reference = pd.read_csv(ROOT/\'predictions.csv\')\\n    blended = reference[[\'image_id\']].copy().set_index(\'image_id\')\\n    blended[\'confidence\'] = 0.0\\n    assert abs(sum(m[\'weight\'] for m in cfg[\'models\']) - 1) < 1e-8\\n    for m in cfg[\'models\']:\\n        frame = pd.read_csv(ROOT/sources[m[\'name\']]).set_index(\'image_id\')\\n        assert frame.index.is_unique and set(frame.index) == set(blended.index)\\n        blended[\'confidence\'] += m[\'weight\'] * frame.loc[blended.index, \'confidence\']\\n    blended[\'label\'] = (blended.confidence >= .5).astype(int)\\n    blended[\'confidence\'] = blended.confidence.round(6)\\n    blended = blended.reset_index()[[\'image_id\', \'label\', \'confidence\']]\\n    audit = compare(pd.read_csv(ROOT/\'outputs/predictions/predictions_original.csv\') if (ROOT/\'outputs/predictions/predictions_original.csv\').exists() else reference, blended, 0)\\n    print(\'\\\\n[1/3] SAVED FIVE-MODEL BLEND:\', \'PASS\' if audit[\'passed\'] else \'FAIL\', flush=True)\\n    print(json.dumps(audit, indent=2), flush=True)\\n    missing = []\\n    for m in cfg[\'models\']:\\n        path = weights_dir/m[\'file\']\\n        if not path.exists() or path.stat().st_size < 1024:\\n            missing.append(str(path))\\n    result = dict(saved_blend=audit, weights_to_csv_status=\'not_run\',\\n                  missing_weights=missing, runs=[],\\n                  reference_sha256=hashlib.sha256((ROOT/\'predictions.csv\').read_bytes()).hexdigest(),\\n                  inference_sha256=hashlib.sha256((ROOT/\'inference_script.py\').read_bytes()).hexdigest())\\n    (OUT/\'verification.json\').write_text(json.dumps(result, indent=2))\\n    if missing or args.test_dir is None:\\n        result[\'weights_to_csv_status\'] = \'blocked\'\\n        result[\'reason\'] = \'Download LFS weights and provide --test-dir with all 3000 official images.\'\\n        print(\'\\\\n[2/3] WEIGHTS-TO-CSV: BLOCKED (not verified)\', flush=True)\\n        print(result[\'reason\'], flush=True)\\n    else:\\n        result[\'device\'] = \'recorded by inference worker\'\\n        result[\'weight_sha256\'] = {m[\'file\']: hashlib.sha256((weights_dir/m[\'file\']).read_bytes()).hexdigest() for m in cfg[\'models\']}\\n        result[\'weights_to_csv_status\'] = \'running\'\\n        (OUT/\'verification.json\').write_text(json.dumps(result, indent=2))\\n        try:\\n            for i in range(args.runs):\\n                target = OUT/f\'predictions_run_{i+1:02}.csv\'\\n                metrics = OUT/f\'metrics_run_{i+1:02}.json\'\\n                command = [sys.executable, \'-u\', str(ROOT/\'inference_script.py\'),\\n                           \'--test_dir\', str(args.test_dir.resolve()), \'--weights\', str(weights_dir/\'ensemble_config.json\'),\\n                           \'--output\', str(target), \'--batch_size\', str(args.batch_size),\\n                           \'--progress_json\', str(OUT/\'progress.json\'), \'--metrics_json\', str(metrics), \'--threads\', str(args.threads)]\\n                print(f\'\\\\n[2/3] REAL INFERENCE RUN {i+1}/{args.runs}\', flush=True)\\n                started = time.perf_counter()\\n                with (OUT/f\'console_run_{i+1:02}.log\').open(\'w\', encoding=\'utf-8\') as log:\\n                    proc = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\\n                    for line in proc.stdout:\\n                        print(line, end=\'\', flush=True)\\n                        log.write(line)\\n                    if proc.wait():\\n                        raise RuntimeError(\'Inference failed; see saved console log\')\\n                measured = json.loads(metrics.read_text(encoding=\'utf-8\'))\\n                result[\'device\'] = measured[\'device_name\']\\n                check = compare(reference, pd.read_csv(target), args.tolerance)\\n                result[\'runs\'].append(dict(run=i+1, process_wall_seconds=time.perf_counter()-started,\\n                                           reproduction=check, metrics=measured))\\n                result[\'weights_to_csv_status\'] = \'passed\' if all(r[\'reproduction\'][\'passed\'] for r in result[\'runs\']) else \'failed\'\\n                (OUT/\'verification.json\').write_text(json.dumps(result, indent=2))\\n        except Exception as error:\\n            result[\'weights_to_csv_status\'] = \'failed\'\\n            result[\'error\'] = str(error)\\n    (OUT/\'verification.json\').write_text(json.dumps(result, indent=2))\\n    # Fresh evidence stays separate; the committed dashboard describes the recorded run.\\n    print(\'\\\\n[3/3] Fresh evidence:\', OUT, flush=True)\\n    print(\'Weights-to-CSV status:\', result[\'weights_to_csv_status\'].upper(), flush=True)\\n    return 0 if result[\'weights_to_csv_status\'] == \'passed\' and audit[\'passed\'] else 2\\n\\n\\nif __name__ == \'__main__\':\\n    sys.exit(main())\\n", "scripts/build_results.py": "\\"\\"\\"Build an offline HTML evidence page and PNG charts from committed results.\\"\\"\\"\\n\\nimport sys as _sys\\nfrom pathlib import Path as _Path\\n_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))\\n\\nimport base64\\nimport html\\nimport json\\nfrom pathlib import Path\\nimport matplotlib\\nmatplotlib.use(\'Agg\')\\nimport matplotlib.pyplot as plt\\nimport pandas as pd\\n\\nROOT = Path(__file__).resolve().parent.parent\\nOUT = ROOT/\'outputs/final_verification\'\\nOUT.mkdir(parents=True, exist_ok=True)\\nplt.rcParams.update({\'font.family\':\'DejaVu Sans\', \'axes.spines.top\':False,\\n                     \'axes.spines.right\':False, \'figure.facecolor\':\'#ffffff\', \'font.size\':11})\\nsections = []\\n\\n\\ndef chart(name, title, explanation, narration):\\n    path = OUT/(name+\'.png\')\\n    plt.savefig(path, dpi=135, bbox_inches=\'tight\')\\n    plt.close()\\n    encoded = base64.b64encode(path.read_bytes()).decode()\\n    sections.append(f\'<section><span class=\\"eyebrow\\">EVIDENCE / {len(sections)+1:02}</span><h2>{title}</h2><p>{explanation}</p><img alt=\\"{title}\\" src=\\"data:image/png;base64,{encoded}\\"><details><summary>What to say in the video</summary><p>{narration}</p></details></section>\')\\n\\n\\ncfg = json.loads((ROOT/\'weights/ensemble_config.json\').read_text(encoding=\'utf-8\'))\\nmodels = cfg[\'models\']\\nfig, ax = plt.subplots(figsize=(9,3.5))\\nax.barh([m[\'name\'] for m in models], [m[\'val_auc\'] for m in models], color=\'#167d8d\')\\nax.set(xlim=(0.5,1), xlabel=\'Saved peak validation AUC (individual models)\')\\nfor i,m in enumerate(models): ax.text(m[\'val_auc\']+.006,i,f\\"{m[\'val_auc\']:.4f}\\",va=\'center\')\\nchart(\'model_auc\',\'Different architectures, one detector\',\\n      \'Individual peak validation values recorded in the ensemble configuration. These are not a measured ensemble AUC or hidden-test scores.\',\\n      \'We compared several architectures on validation data. EfficientNetV2-S has the highest recorded individual AUC, 0.9025. We combine five models using fixed probability weights. Model diversity motivates this design, but an improvement must be measured rather than assumed.\')\\n\\nfig, ax = plt.subplots(figsize=(9,3.5))\\nfor label,path in [(\'MaxViT\',\'outputs/train_log_maxvit.csv\'),(\'ConvNeXt\',\'outputs/train_log_convnext.csv\')]:\\n    df=pd.read_csv(ROOT/path)\\n    ax.plot(df.epoch,df.val_auc,label=label+\' validation\',linewidth=2)\\n    ax.plot(df.epoch,df.train_auc,label=label+\' training\',linestyle=\'--\',alpha=.65)\\nax.set(xlabel=\'Epoch\',ylabel=\'AUC\',ylim=(.5,1.01)); ax.legend(ncol=2)\\nchart(\'learning\',\'Why checkpoint selection matters\',\\n      \'Training and validation histories from saved logs. Peak validation checkpoints are preferable to selecting the last epoch automatically.\',\\n      \'Training AUC continues rising while validation performance levels off. This gap indicates overfitting. We select checkpoints using validation performance, and keep the official test images out of training.\')\\n\\nstats=pd.read_csv(ROOT/\'outputs/robustness/statistics.csv\')\\nfig, ax=plt.subplots(figsize=(9,3.8))\\nax.errorbar(range(len(stats)),stats.auc,yerr=[stats.auc-stats.auc_ci_low,stats.auc_ci_high-stats.auc],fmt=\'o\',capsize=5,color=\'#167d8d\')\\nax.set(xticks=range(len(stats)),xticklabels=[\'Clean\',\'JPEG 50\',\'JPEG 70\',\'Blur\',\'Noise\'],ylabel=\'AUC with saved 95% bootstrap intervals\',ylim=(.75,1))\\nchart(\'robustness\',\'Robustness, with uncertainty visible\',\\n      \'Existing FOUR-model ensemble; 200 balanced validation images. Clean AUC 0.8813; largest observed drop 0.0129. Intervals do not prove the population drop stays below 0.02. This does not validate the final five-model ensemble.\',\\n      \'We tested the earlier four-model ensemble on 200 balanced validation images. Noise produced the largest observed AUC drop, 0.0129. The uncertainty intervals matter: this sample does not guarantee the same robustness on unseen data or for our final five-model configuration.\')\\n\\ntiming=pd.read_csv(ROOT/\'outputs/efficiency_t4/summary.csv\')\\nfig, ax=plt.subplots(figsize=(9,3.8))\\nax.errorbar(range(len(stats)),stats.auc_drop,yerr=[stats.auc_drop-stats.drop_ci_low,stats.drop_ci_high-stats.auc_drop],fmt=\'o\',capsize=5,color=\'#167d8d\')\\nax.axhline(.02,color=\'#bc633e\',linestyle=\'--\',label=\'0.02 analysis target (not official scoring formula)\')\\nax.axhline(0,color=\'gray\',linewidth=.7)\\nax.set(xticks=range(len(stats)),xticklabels=[\'Clean\',\'JPEG 50\',\'JPEG 70\',\'Blur\',\'Noise\'],ylabel=\'Paired AUC drop with 95% bootstrap intervals\');ax.legend(fontsize=9)\\nchart(\'auc_drop\',\'Observed drops versus statistical uncertainty\',\\n      \'Earlier FOUR-model sample. Negative drops indicate higher measured AUC than clean. Noise and blur intervals extend above 0.02; a small observed drop does not prove a strict upper bound.\',\\n      \'The dots are observed changes relative to clean images. The intervals capture sample uncertainty. Noise and blur extend beyond the analysis target, so we report encouraging observed stability without claiming a guarantee.\')\\n\\nfrom sklearn.metrics import roc_curve,confusion_matrix,roc_auc_score\\nold=pd.read_csv(ROOT/\'outputs/robustness/predictions_ensemble_clean.csv\')\\ntruth=\'label\' if \'label\' in old else \'true_label\'\\nprob=\'confidence\' if \'confidence\' in old else \'probability\'\\nif truth in old and prob in old:\\n    assert abs(roc_auc_score(old[truth],old[prob])-stats.iloc[0].auc)<1e-6\\n    fig,(ax,bx)=plt.subplots(1,2,figsize=(10,4))\\n    fpr,tpr,_=roc_curve(old[truth],old[prob]);ax.plot(fpr,tpr,color=\'#167d8d\');ax.plot([0,1],[0,1],\'--\',color=\'gray\')\\n    ax.set(xlabel=\'False positive rate\',ylabel=\'True positive rate\',title=\'Earlier four-model ROC\')\\n    cm=confusion_matrix(old[truth],old[prob]>=.5,labels=[0,1]);bx.imshow(cm,cmap=\'Blues\')\\n    for i in range(2):\\n        for j in range(2):bx.text(j,i,str(cm[i,j]),ha=\'center\',va=\'center\')\\n    bx.set(xticks=[0,1],yticks=[0,1],xticklabels=[\'Real\',\'Fake\'],yticklabels=[\'Real\',\'Fake\'],xlabel=\'Predicted\',ylabel=\'True\',title=\'Clean sample: threshold 0.5\')\\n    chart(\'reference_roc\',\'Ranking performance and classification errors\',\\n          \'Existing FOUR-model balanced validation sample, 200 images. ROC and confusion counts are recomputed from the saved predictions, with AUC checked against statistics.csv.\',\\n          \'The ROC curve evaluates ranking across thresholds. The confusion matrix shows actual correct and incorrect decisions at threshold 0.5. These are earlier four-model validation results, not private test results.\')\\n\\nfig, ax=plt.subplots(figsize=(9,3.3))\\nax.bar([\'Run 1\',\'Run 2\',\'Run 3\'],timing.wall_seconds,color=\'#167d8d\')\\nax.axhline(300,color=\'#bc633e\',linestyle=\'--\',label=\'5-minute scoring boundary\')\\nax.set(ylabel=\'End-to-end seconds (Tesla T4)\',ylim=(0,330));ax.legend()\\nchart(\'timing\',\'Measured speed of the earlier pipeline\',\\n      \'Supplied FOUR-model T4 measurements: 77.66, 60.36, 56.74 seconds. Median 60.36 seconds. Includes process startup and CSV writing; excludes downloads. Final five-model timing remains separate.\',\\n      \'The earlier four-model pipeline processed all 3,000 images in a median of 60.36 seconds on a T4. We do not transfer that timing to the five-model pipeline. The final runner records its own runtime and memory when executed on the target hardware.\')\\n\\npred=pd.read_csv(ROOT/\'predictions.csv\')\\nfig, ax=plt.subplots(figsize=(9,3.3))\\nax.hist(pred.confidence,bins=25,color=\'#167d8d\',edgecolor=\'white\');ax.axvline(.5,color=\'#bc633e\',linestyle=\'--\')\\nax.set(xlabel=\'Predicted probability of Fake\',ylabel=\'Number of test images\')\\nchart(\'confidence\',\'What the submitted predictions contain\',\\n      f\'{len(pred):,} unlabelled test predictions. This distribution describes model outputs; it cannot measure test accuracy, AUC, or calibration without labels.\',\\n      \'Each image receives a probability of being fake, and the label uses a threshold of 0.5. The histogram shows how those probabilities are distributed. Because test labels are private, this graph does not establish test accuracy or calibration.\')\\n\\nverification=json.loads((OUT/\'verification.json\').read_text(encoding=\'utf-8\')) if (OUT/\'verification.json\').exists() else {}\\nif (ROOT/\'outputs/final_validation/protocol.json\').exists():\\n    final=pd.read_csv(ROOT/\'outputs/final_validation/statistics.csv\')\\n    fig,ax=plt.subplots(figsize=(9,3.5))\\n    ax.errorbar(range(len(final)),final.auc,yerr=[final.auc-final.auc_ci_low,final.auc_ci_high-final.auc],fmt=\'o\',capsize=5,color=\'#167d8d\')\\n    ax.set(xticks=range(len(final)),xticklabels=final.condition, ylabel=\'Five-model AUC with 95% bootstrap intervals\')\\n    ax.tick_params(axis=\'x\',labelrotation=15)\\n    chart(\'final_robustness\',\'Final five-model validation and robustness\',\\n          \'Measured on the existing fixed 200-image balanced validation sample; 2,000 stratified bootstrap resamples. Not hidden-test performance.\',\\n          \'This graph evaluates the final five-model pipeline on the same fixed validation sample. We report uncertainty alongside AUC, and keep this separate from hidden-test performance.\')\\n    clean=pd.read_csv(ROOT/\'outputs/final_validation/predictions_clean.csv\')\\n    from sklearn.metrics import roc_curve,confusion_matrix\\n    fig,(ax,bx)=plt.subplots(1,2,figsize=(10,4))\\n    fpr,tpr,_=roc_curve(clean.label,clean.confidence)\\n    ax.plot(fpr,tpr,color=\'#167d8d\');ax.plot([0,1],[0,1],\'--\',color=\'gray\')\\n    ax.set(xlabel=\'False positive rate\',ylabel=\'True positive rate\',title=\'Clean validation ROC\')\\n    cm=confusion_matrix(clean.label,clean.confidence>=.5,labels=[0,1]);bx.imshow(cm,cmap=\'Blues\')\\n    for i in range(2):\\n        for j in range(2):bx.text(j,i,str(cm[i,j]),ha=\'center\',va=\'center\')\\n    bx.set(xticks=[0,1],yticks=[0,1],xticklabels=[\'Real\',\'Fake\'],yticklabels=[\'Real\',\'Fake\'],xlabel=\'Predicted\',ylabel=\'True\',title=\'Threshold = 0.5\')\\n    chart(\'final_roc\',\'Ranking quality and concrete errors\',\\n          \'ROC uses continuous fake probabilities. The confusion matrix uses threshold 0.5 on the same 200 validation images.\',\\n          \'ROC describes the trade-off across thresholds, while the confusion matrix counts errors at our fixed threshold of 0.5. These are validation results and do not reveal private test labels.\')\\nstatus=verification.get(\'weights_to_csv_status\',\'not_run\')\\nif verification.get(\'runs\'):\\n    fig,ax=plt.subplots(figsize=(9,3.3))\\n    ax.bar([str(r[\'run\']) for r in verification[\'runs\']],[r[\'process_wall_seconds\'] for r in verification[\'runs\']],color=\'#167d8d\')\\n    ax.set(xlabel=\'Final five-model run\',ylabel=\'Process wall seconds\')\\n    chart(\'final_timing\',\'Final five-model measured runtime\',html.escape(verification.get(\'device\',\'Unknown device\'))+\'; consult verification.json for per-run reproduction results.\',\\n          \'These timings come from the final five-model submission script. The device is identified explicitly; CPU measurements are not T4 benchmark results.\')\\n\\ngradcams=[]\\nfor p in list((ROOT/\'outputs/gradcam\').glob(\'*.png\'))[:4]:\\n    gradcams.append(\'<img alt=\\"Grad-CAM attention overlay\\" src=\\"data:image/png;base64,\'+base64.b64encode(p.read_bytes()).decode()+\'\\">\')\\npage=\'\'\'<!doctype html><html lang=\\"en\\"><meta charset=\\"utf-8\\"><meta name=\\"viewport\\" content=\\"width=device-width,initial-scale=1\\"><title>SMPS | Deepfake detection evidence</title><style>\\n:root{color-scheme:light}*{box-sizing:border-box}body{margin:0;background:#f2f3ee;color:#152c35;font:17px/1.6 system-ui,sans-serif}header{background:#153b46;color:#fff;padding:65px max(6vw,24px)}header h1{font-size:clamp(35px,5vw,66px);line-height:1.08;max-width:900px;margin:18px 0}header p{max-width:780px;color:#d6e8e6}.eyebrow{font-size:12px;letter-spacing:.18em;font-weight:700;color:#167d8d}header .eyebrow{color:#a9e4d8}nav{display:flex;gap:24px;padding:18px 6vw;background:#fff;position:sticky;top:0;z-index:2;box-shadow:0 1px 0 #ddd}a{color:#167d8d}main{max-width:1120px;margin:auto;padding:30px 24px}section{padding:32px;margin:24px 0;background:white;border-radius:8px}h2{font-size:29px;line-height:1.2}section>img{width:100%;height:auto}details{background:#edf5f2;padding:16px;margin-top:14px}summary{cursor:pointer;font-weight:650}.metrics{display:grid;grid-template-columns:repeat(3,1fr);gap:18px}.metric{border-left:3px solid #167d8d;padding:12px}.metric strong{display:block;font-size:30px}.warning{border-left:4px solid #bc633e;padding:16px;background:#fff4e9}pre{white-space:pre-wrap;overflow-wrap:anywhere;background:#102d36;color:#ddf2e8;padding:18px;font-size:14px}.gallery{display:grid;grid-template-columns:1fr 1fr;gap:15px}.gallery img{width:100%}button{background:#167d8d;color:white;border:0;padding:12px 18px;cursor:pointer;font:inherit}@media(max-width:650px){section{padding:20px}.metrics{grid-template-columns:1fr}.gallery{grid-template-columns:1fr}nav{position:static;font-size:14px;gap:15px}}@media print{nav,button{display:none}section{break-inside:avoid}details{display:block}}\\n</style><header><span class=\\"eyebrow\\">TEAM SMPS / IIIT BANGALORE HACKATHON</span><h1>Detecting synthetic faces.<br>Showing the evidence.</h1><p>Five architectures, fixed probability blending, and a clear distinction between completed measurements and verification still to run.</p><button onclick=\\"document.documentElement.requestFullscreen?.()\\">Presentation mode</button></header><nav><a href=\\"#status\\">Run status</a><a href=\\"#evidence\\">Results & graphs</a><a href=\\"#explain\\">Explainability</a><a href=\\"#run\\">Run commands</a></nav><main>\'\'\'\\npage+=f\'<section id=\\"status\\"><span class=\\"eyebrow\\">VERIFICATION</span><h2>Submission readiness</h2><div class=\\"metrics\\"><div class=\\"metric\\"><strong>3,000</strong>Unique prediction IDs</div><div class=\\"metric\\"><strong>5</strong>Ensemble models</div><div class=\\"metric\\"><strong>{html.escape(status.upper())}</strong>Weights-to-CSV reproduction</div></div><p class=\\"warning\\">Saved CSV blending is a separate check from fresh inference. No official score is known. Four-model robustness and T4 evidence must not be described as five-model results.</p><p id=\\"progress\\">Live progress appears here when served over localhost.</p><pre>{html.escape(json.dumps(verification,indent=2))}</pre></section><div id=\\"evidence\\"></div>\'\\npage+=\'\'.join(sections)\\npage+=\'<section id=\\"explain\\"><span class=\\"eyebrow\\">EXPLAINABILITY</span><h2>Where the detector attends</h2><div class=\\"gallery\\">\'+\'\'.join(gradcams)+\'</div><p>Grad-CAM highlights regions influencing a prediction. It is a qualitative diagnostic, not proof of causality or correctness.</p><details><summary>What to say in the video</summary><p>These overlays help inspect the regions influencing the detector. We use them to look for reliance on facial regions and possible shortcuts. A heatmap alone cannot prove that a prediction is correct.</p></details></section>\'\\npage+=\'\'\'<section id=\\"run\\"><h2>Reproduce, compare, present</h2><pre>python scripts/verify_final.py\\n# After downloading the LFS assets:\\npython scripts/verify_final.py --test-dir datasets/extracted/hackathon_dataset/test/images --runs 3\\n# Serve this page from a second terminal:\\npython -m http.server 8765 --bind 127.0.0.1\\n# Open http://127.0.0.1:8765/presentation/results.html</pre><p>Audit-only mode exits with code 2 when fresh inference is blocked. It never reports a pass without running real inference. Test predictions are not overwritten.</p></section></main><script>\\nif(location.protocol!==\'file:\'){let last=\'\';setInterval(async()=>{try{let r=await fetch(\'outputs/final_verification/progress.json?t=\'+Date.now());if(r.ok){let p=await r.json();document.querySelector(\'#progress\').textContent=\'LIVE: \'+p.stage+\' | \'+p.processed+\' / \'+p.total+\' images\'+(p.device?\' | \'+p.device:\'\')}}catch{}try{let r=await fetch(\'outputs/final_verification/verification.json?t=\'+Date.now());if(r.ok){let s=await r.text();if(last&&last!==s)location.reload();last=s}}catch{}},2500)}\\n</script></html>\'\'\'\\n(ROOT/\'presentation/results.html\').write_text(page,encoding=\'utf-8\')\\nprint(\'[GRAPHS] Built offline presentation/results.html and PNG figures in outputs/final_verification\')\\n\\n# Keep the public dashboard presentation-only after chart regeneration.\\nimport runpy\\nrunpy.run_path(str(ROOT/\'scripts/render_dashboard.py\'))\\n", "scripts/validate_final.py": "\\"\\"\\"Five-model validation/robustness on the existing fixed 200-image sample.\\"\\"\\"\\n\\nimport sys as _sys\\nfrom pathlib import Path as _Path\\n_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))\\n\\nimport argparse\\nimport json\\nimport gc\\nfrom pathlib import Path\\nimport numpy as np\\nimport pandas as pd\\nfrom PIL import Image\\nimport torch\\nfrom sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, roc_curve\\nfrom inference_script import load_model, get_inference_transforms, EnsembleRunner\\nfrom robustness_test import perturb\\n\\nROOT = Path(__file__).resolve().parent.parent\\n\\n\\ndef main():\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument(\'--dataset-root\',type=Path,required=True)\\n    parser.add_argument(\'--batch-size\',type=int,default=16)\\n    parser.add_argument(\'--output-dir\',type=Path,default=ROOT/\'outputs/reproduction/validation\')\\n    parser.add_argument(\'--weights-dir\',type=Path,default=ROOT/\'weights\')\\n    args=parser.parse_args()\\n    out=args.output_dir.resolve()\\n    if out == (ROOT/\'outputs/final_validation\').resolve():\\n        parser.error(\'Recorded evidence is immutable; choose a new output directory.\')\\n    if (out/\'protocol.json\').exists():\\n        parser.error(\'Output already contains a run; choose a new --output-dir.\')\\n    out.mkdir(parents=True,exist_ok=True)\\n    sample=pd.read_csv(ROOT/\'outputs/robustness/sample_manifest.csv\')\\n    val=pd.read_csv(ROOT/\'data/val_split.csv\');train=pd.read_csv(ROOT/\'data/train_split.csv\')\\n    assert not set(train.image_id)&set(val.image_id)\\n    assert set(sample.image_id)<=set(val.image_id) and sample.image_id.is_unique\\n    metadata=pd.read_csv(args.dataset_root/\'train_metadata.csv\').set_index(\'image_id\')\\n    assert all(int(metadata.loc[r.image_id,\'label\'])==r.label for r in sample.itertuples())\\n    device=\'cuda\' if torch.cuda.is_available() else \'cpu\'\\n    torch.set_num_threads(4)\\n    config=json.loads((args.weights_dir/\'ensemble_config.json\').read_text(encoding=\'utf-8\'))\\n    tf=get_inference_transforms(); rows=[]\\n    y=sample.label.to_numpy()\\n    rng=np.random.default_rng(42)\\n    indices=[np.concatenate([rng.choice(np.flatnonzero(y==c),int((y==c).sum()),replace=True) for c in (0,1)]) for _ in range(2000)]\\n    clean=None\\n    conditions=[\'clean\',\'jpeg_q50\',\'jpeg_q70\',\'blur_sigma1.5\',\'noise_sigma15\']\\n    blended={condition:np.zeros(len(sample),dtype=np.float32) for condition in conditions}\\n    with torch.inference_mode():\\n      for info in config[\'models\']:\\n        single=load_model(str(args.weights_dir/info[\'file\']),info[\'arch\'],device)\\n        model=EnsembleRunner([(single,1.0)],device)\\n        for condition in conditions:\\n            probs=[]\\n            for start in range(0,len(sample),args.batch_size):\\n                tensors=[]\\n                for row in sample.iloc[start:start+args.batch_size].itertuples():\\n                    path=args.dataset_root/\'train\'/(\'fake\' if row.label else \'real\')/(row.image_id+\'.jpg\')\\n                    array=np.asarray(Image.open(path).convert(\'RGB\'))\\n                    tensors.append(tf(Image.fromarray(perturb(array,condition,row.image_id))))\\n                probs.extend(model(torch.stack(tensors).to(device)).cpu().tolist())\\n                print(f\'[FINAL VALIDATION] {info[\\"name\\"]} / {condition}: {min(start+args.batch_size,len(sample))}/{len(sample)}\',flush=True)\\n            blended[condition]+=info[\'weight\']*np.asarray(probs,dtype=np.float32)\\n        del model,single\\n        gc.collect()\\n        if device==\'cuda\':torch.cuda.empty_cache()\\n      for condition in conditions:\\n            p=blended[condition]\\n            if condition==\'clean\':clean=p.copy()\\n            auc=roc_auc_score(y,p)\\n            boot=np.array([roc_auc_score(y[ix],p[ix]) for ix in indices])\\n            drops=np.array([roc_auc_score(y[ix],clean[ix])-roc_auc_score(y[ix],p[ix]) for ix in indices])\\n            tn,fp,fn,tp=confusion_matrix(y,p>=.5,labels=[0,1]).ravel()\\n            lo,hi=np.quantile(boot,[.025,.975]);dl,dh=np.quantile(drops,[.025,.975])\\n            rows.append(dict(condition=condition,n=len(y),auc=auc,auc_ci_low=lo,auc_ci_high=hi,\\n                             auc_drop=roc_auc_score(y,clean)-auc,drop_ci_low=dl,drop_ci_high=dh,\\n                             f1=f1_score(y,p>=.5),tn=int(tn),fp=int(fp),fn=int(fn),tp=int(tp)))\\n            frame=sample[[\'image_id\',\'label\']].copy();frame[\'confidence\']=p\\n            frame.to_csv(out/f\'predictions_{condition}.csv\',index=False)\\n            pd.DataFrame(rows).to_csv(out/\'statistics.csv\',index=False)\\n    (out/\'protocol.json\').write_text(json.dumps(dict(models=5,sample=200,seed=42,bootstrap=2000,\\n        device=device,precision=\'float32\',tta=\'original+horizontal_flip\',\\n        limitation=\'Fixed validation subset, not full validation or hidden test. FP32 validation; submission default inference uses FP32.\'),indent=2))\\n    import subprocess,sys\\n    print(\'Fresh validation evidence:\',out)\\n\\n\\nif __name__==\'__main__\':main()\\n", "scripts/prepare_final_delivery.py": "\\"\\"\\"Generate the concise report and self-contained final-run Colab notebook.\\"\\"\\"\\n\\nimport sys as _sys\\nfrom pathlib import Path as _Path\\n_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))\\n\\nimport json\\nimport shutil\\nimport sys\\nfrom pathlib import Path\\nfrom reportlab.lib import colors\\nfrom reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle\\nfrom reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image, PageBreak\\nfrom reportlab.lib.pagesizes import A4\\n\\nif \'--rebuild-notebook\' in sys.argv:\\n    raise SystemExit(\'The final verification notebook is maintained directly; use scripts/check_delivery.py to validate it.\')\\n\\nROOT=Path(__file__).resolve().parent.parent\\nverification_path=ROOT/\'outputs/final_verification/verification.json\'\\nverification=json.loads(verification_path.read_text(encoding=\'utf-8\')) if verification_path.exists() else {}\\nfresh_status=verification.get(\'weights_to_csv_status\',\'not_run\')\\nstyles=getSampleStyleSheet()\\nstyles.add(ParagraphStyle(name=\'BodyCompact\',fontName=\'Helvetica\',fontSize=9,leading=12,spaceAfter=7))\\nstory=[]\\ndef para(text,style=\'BodyCompact\'):story.append(Paragraph(text,styles[style]))\\ndef image(path,w=475,h=170):story.append(Image(str(ROOT/path),width=w,height=h,kind=\'proportional\'))\\nbackup=ROOT/\'docs/reports/report_detailed.pdf\'\\nif not backup.exists():shutil.copy2(ROOT/\'docs/reports/report.pdf\',backup)\\npara(\'Deepfake face classification\',\'Title\')\\npara(\'Team SMPS | IIIT Bangalore Hackathon | Submission technical report\')\\npara(\'1. Objective, data and validation\',\'Heading2\')\\npara(\'Predict whether an aligned 224 x 224 face image is real (0) or fake (1). The supplied dataset has 7,000 labelled training images and 3,000 unlabelled test images. The saved stratified split uses 5,600 training and 1,400 validation images. Only the supplied images are used for fine-tuning; ImageNet initialization is permitted. Validation AUC guides checkpoint selection. Private test AUC is unknown. Ethical use requires acknowledging false positives and avoiding treating a detector output as definitive evidence.\')\\npara(\'2. Architecture and model selection\',\'Heading2\')\\npara(\'The submission averages original and horizontal-flip probabilities per model, then blends five architectures. Fixed weights are heuristics informed by individual validation performance; they are not claimed to be optimized on hidden labels. A diverse model set does not guarantee an ensemble gain.\')\\nrows=[[\'Architecture\',\'Saved peak val AUC\',\'Blend weight\'],[\'EfficientNetV2-S\',\'0.9025\',\'0.40\'],[\'ConvNeXt-Tiny\',\'0.8973\',\'0.25\'],[\'Swin-Tiny\',\'0.8953\',\'0.15\'],[\'MaxViT-Tiny\',\'0.8902\',\'0.12\'],[\'DenseNet-121\',\'0.8905\',\'0.08\']]\\nt=Table(rows,colWidths=[255,125,95]);t.setStyle(TableStyle([(\'BACKGROUND\',(0,0),(-1,0),colors.HexColor(\'#153b46\')),(\'TEXTCOLOR\',(0,0),(-1,0),colors.white),(\'FONTSIZE\',(0,0),(-1,-1),9),(\'BOTTOMPADDING\',(0,0),(-1,-1),7),(\'TOPPADDING\',(0,0),(-1,-1),7),(\'LINEBELOW\',(0,1),(-1,-1),.3,colors.lightgrey)]));story.append(t)\\npara(\'3. Training pipeline and challenges\',\'Heading2\')\\npara(\'Recorded training uses ImageNet normalization, geometric and photometric augmentation, compression/noise augmentation, AdamW and scheduled learning rates. Most candidate notebooks use binary focal loss (gamma 1.5). Exact settings differ by architecture and are retained in the training notebooks. Training AUC approaching 1 while validation plateaus motivates early checkpoint selection. EVA-02 Tiny recorded AUC 0.7126; data size and architecture are possible explanations, not experimentally isolated causes.\')\\nimage(\'outputs/final_verification/learning.png\',475,165)\\nstory.append(PageBreak())\\npara(\'4. Evaluation, uncertainty and explainability\',\'Heading2\')\\nif (ROOT/\'outputs/final_validation/statistics.csv\').exists():\\n    import csv\\n    records=list(csv.DictReader((ROOT/\'outputs/final_validation/statistics.csv\').open()))\\n    clean=next(r for r in records if r[\'condition\']==\'clean\')\\n    worst=max(records,key=lambda r:float(r[\'auc_drop\']))\\n    para(f\\"Final five-model FP32 evaluation uses 200 fixed balanced validation images. Clean AUC is {float(clean[\'auc\']):.4f} (95% bootstrap interval {float(clean[\'auc_ci_low\']):.4f}-{float(clean[\'auc_ci_high\']):.4f}). Largest observed AUC drop is {float(worst[\'auc_drop\']):.4f} under {worst[\'condition\']}, with paired interval {float(worst[\'drop_ci_low\']):.4f}-{float(worst[\'drop_ci_high\']):.4f}. Intervals use 2,000 stratified resamples; these are validation-subset estimates, not hidden-test scores or population guarantees.\\")\\nelse:\\n    para(\'Earlier four-model robustness evidence uses 200 fixed validation images, balanced by class. Clean AUC is 0.8813; JPEG Q50 0.8874; JPEG Q70 0.8854; blur 0.8740; noise 0.8684. The largest observed absolute drop is 0.0129. Bootstrap intervals for some drops extend beyond 0.02, so this sample does not establish a population guarantee. These measurements are not five-model robustness results.\')\\nimage(\'outputs/final_verification/final_robustness.png\' if (ROOT/\'outputs/final_verification/final_robustness.png\').exists() else \'outputs/final_verification/robustness.png\',475,150)\\npara(\'Grad-CAM overlays are included as qualitative diagnostics. They highlight regions influencing predictions, without proving correct reasoning, causal reliance on a region, or generalization to new manipulation methods.\')\\npara(\'5. Efficiency and reproducibility status\',\'Heading2\')\\npara(\'Supplied four-model Tesla T4 runs took 77.66, 60.36 and 56.74 seconds for 3,000 images (median 60.36 seconds). The final five-model CLI is a different pipeline. Official test performance and scores remain unknown.\')\\nif verification.get(\'runs\'):\\n    timings=\', \'.join(f\'{r[\\"process_wall_seconds\\"]:.2f}\' for r in verification[\'runs\'])\\n    para(f\'Final five-model measured process-wall times: {timings} seconds on {verification.get(\\"device\\",\\"unknown device\\")}. Fresh weights-to-CSV status: <b>{fresh_status.upper()}</b>. Detailed per-run probability differences, label mismatches and the comparison tolerance are in verification.json. CPU results must not be presented as T4 measurements.\')\\nelse:\\n    para(f\'Fresh weights-to-CSV status: <b>{fresh_status.upper()}</b>. The local attempt encountered resource constraints; no completed five-model timing is available. The prepared Colab notebook performs this check on a T4, saves evidence, and regenerates this report from completed results.\')\\npara(\'The legacy five-model blend reproduces its original CSV exactly. The original is preserved separately when a fresh FP32 submission candidate is generated. Final acceptance requires an independent weights-to-CSV replay against the fresh candidate. scripts/verify_final.py performs the latter with a declared tolerance, logs, source/weight hashes and actual device/runtime evidence. A mismatch must be resolved before submission. The HTML page identifies pending checks explicitly.\')\\npara(\'6. Reproduction commands\',\'Heading2\')\\npara(\'<font face=\\"Courier\\" size=\\"8\\">git lfs pull --include=\\"weights/*,datasets/hackathon_dataset.zip\\"<br/>python inference_script.py --test_dir PATH/test/images --weights weights/ensemble_config.json --output fresh.csv<br/>python scripts/verify_final.py --test-dir PATH/test/images --runs 3<br/>python scripts/validate_final.py --dataset-root PATH<br/>python scripts/build_results.py</font>\')\\npara(\'Install requirements.txt in Python 3.10 for the original training environment. The final-run Colab notebook records its actual runtime dependencies; preserve those records with generated evidence. Extract the official archive and pass paths explicitly. The official participant template uses image_id,label,confidence and inference_script.py. Confidence is probability of Fake. Required code, configuration and all five binary weight files must accompany the submission.\')\\npara(\'Limitations: hidden test labels unavailable; robustness uses a validation subset; no demonstrated final five-model AUC improvement; hardware and precision can alter probabilities. Official points and bonus awards are determined by judges.\')\\ndef footer(canvas,doc):\\n    canvas.setFont(\'Helvetica\',8);canvas.setFillColor(colors.grey)\\n    canvas.drawString(42,24,\'TEAM SMPS | Evidence and limitations stated separately\')\\n    canvas.drawRightString(A4[0]-42,24,str(doc.page))\\nSimpleDocTemplate(str(ROOT/\'docs/reports/report.pdf\'),pagesize=A4,rightMargin=42,leftMargin=42,topMargin=32,bottomMargin=36).build(story,onFirstPage=footer,onLaterPages=footer)\\nif \'--report-only\' in sys.argv or \'--rebuild-notebook\' not in sys.argv:\\n    print(\'Updated two-page report from actual verification evidence.\')\\n    sys.exit(0)\\n\\ncells=[]\\ndef md(s):cells.append(dict(cell_type=\'markdown\',metadata={},source=s.splitlines(True)))\\ndef code(s):cells.append(dict(cell_type=\'code\',metadata={},execution_count=None,outputs=[],source=s.splitlines(True)))\\nmd(\'# SMPS final five-model verification\\\\nChoose **Runtime > Change runtime type > T4 GPU**, then Run all. This notebook embeds the updated local scripts, downloads only official assets, preserves submitted predictions, records failures honestly, and downloads evidence. Existing four-model evidence is never relabelled as five-model evidence.\')\\ncode(\\"import torch\\\\nassert torch.cuda.is_available(), \'Select a T4 GPU runtime\'\\\\nassert \'T4\' in torch.cuda.get_device_name(), \'This benchmark requires a T4\'\\\\nprint(torch.cuda.get_device_name())\\\\n\\")\\ncode(\'%pip -q install timm==1.0.31 pandas matplotlib scikit-learn opencv-python-headless reportlab pypdf pymupdf\\\\n\'+(ROOT/\'scripts/colab_setup.py\').read_text(encoding=\'utf-8\')+\'\\\\nimport subprocess\\\\n\')\\nembedded={name:(ROOT/name).read_text(encoding=\'utf-8\') for name in [\'inference_script.py\',\'ensemble_worker_control.py\',\'scripts/verify_final.py\',\'scripts/build_results.py\',\'scripts/validate_final.py\',\'scripts/prepare_final_delivery.py\',\'scripts/colab_setup.py\']}\\ncode(\'import json\\\\nscripts=json.loads(\'+repr(json.dumps(embedded))+\')\\\\nfor name, source in scripts.items():\\\\n    Path(name).parent.mkdir(parents=True, exist_ok=True)\\\\n    Path(name).write_text(source, encoding=\\"utf-8\\")\\\\nprint(\\"Installed exact final local scripts\\")\\\\n\')\\ncode(\\"import zipfile, hashlib\\\\nextract=project/\'datasets/extracted\'\\\\nextract.mkdir(exist_ok=True)\\\\nwith zipfile.ZipFile(\'datasets/hackathon_dataset.zip\') as z:\\\\n    for item in z.infolist():\\\\n        target=(extract/item.filename).resolve()\\\\n        assert target.is_relative_to(extract.resolve())\\\\n        if not item.is_dir():\\\\n            target.parent.mkdir(parents=True,exist_ok=True)\\\\n            target.write_bytes(z.read(item))\\\\nmetadata=list(extract.rglob(\'train_metadata.csv\'))\\\\nassert len(metadata)==1, \'Expected one official dataset\'\\\\ndataset=metadata[0].parent\\\\nprint(\'Dataset:\',dataset)\\\\n\\")\\ncode(\\"# Preserve both the inference exit status and all diagnostic evidence.\\\\nrun=subprocess.run([\'python\',\'-u\',\'scripts/verify_final.py\',\'--test-dir\',str(dataset/\'test/images\'),\'--runs\',\'3\'])\\\\nprint(\'Verification exit code:\',run.returncode)\\\\n# Code 2 means blocked or mismatched reproduction, never a pass.\\\\n\\")\\ncode(\\"# Measure final five-model validation / robustness with uncertainty.\\\\nsubprocess.run([\'python\',\'-u\',\'scripts/validate_final.py\',\'--dataset-root\',str(dataset)],check=True)\\\\nwith open(\'outputs/final_verification/environment.txt\',\'w\') as env_log:\\\\n    subprocess.run([\'python\',\'-m\',\'pip\',\'freeze\'],stdout=env_log,check=True)\\\\nsubprocess.run([\'python\',\'scripts/build_results.py\'],check=True)\\\\nsubprocess.run([\'python\',\'scripts/prepare_final_delivery.py\',\'--report-only\'],check=True)\\\\nfrom pypdf import PdfReader\\\\nassert len(PdfReader(\'docs/reports/report.pdf\').pages)==2, \'Check report layout before submission\'\\\\nfrom IPython.display import display, HTML\\\\ndisplay(HTML(Path(\'presentation/results.html\').read_text(encoding=\'utf-8\')))\\\\n\\")\\ncode(\\"import zipfile\\\\nfrom google.colab import files\\\\nwith zipfile.ZipFile(\'/content/SMPS_final_evidence.zip\',\'w\',zipfile.ZIP_DEFLATED) as z:\\\\n    z.write(\'presentation/results.html\')\\\\n    z.write(\'docs/reports/report.pdf\')\\\\n    for name in scripts:z.write(name)\\\\n    for folder in [\'outputs/final_verification\',\'outputs/final_validation\']:\\\\n        for path in Path(folder).rglob(\'*\'):\\\\n            if path.is_file():z.write(path)\\\\nfiles.download(\'/content/SMPS_final_evidence.zip\')\\\\nprint(Path(\'outputs/final_verification/verification.json\').read_text(encoding=\'utf-8\'))\\\\nprint(\'Do not submit until weights_to_csv_status is passed. A failed comparison needs investigation.\')\\\\n\\")\\n(ROOT/\'notebooks/SMPS_Final_Five_Model_Verification.ipynb\').write_text(json.dumps(dict(cells=cells,metadata=dict(kernelspec=dict(display_name=\'Python 3\',language=\'python\',name=\'python3\'),accelerator=\'GPU\'),nbformat=4,nbformat_minor=5),indent=1))\\nprint(\'Generated docs/reports/report.pdf and final-run Colab notebook; detailed report retained.\')\\n", "scripts/colab_setup.py": "\\"\\"\\"Colab asset setup without depending on Git LFS installation.\\"\\"\\"\\n\\n\\nimport hashlib\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport urllib.request\\nimport zipfile\\n\\nsource_revision = os.environ.get(\'SMPS_SOURCE_REVISION\', \'a63df93271c87ca749643a30c4bc188d207f2c31\')\\nproject = Path(os.environ.get(\'SMPS_COLAB_ROOT\', \'/content/SMPS_FINAL\'))\\nproject.mkdir(parents=True, exist_ok=True)\\nif not (project/\'weights/ensemble_config.json\').exists():\\n    archive = Path(\'/content/smps_source.zip\')\\n    print(\'[SETUP] Downloading public project source\', flush=True)\\n    urllib.request.urlretrieve(\'https://codeload.github.com/snehav-Unique/SMPS_MODEL/zip/\'+source_revision, archive)\\n    with zipfile.ZipFile(archive) as bundle:\\n        for entry in bundle.infolist():\\n            relative = Path(*Path(entry.filename).parts[1:])\\n            target = (project/relative).resolve()\\n            assert target.is_relative_to(project.resolve())\\n            if not entry.is_dir():\\n                target.parent.mkdir(parents=True, exist_ok=True)\\n                target.write_bytes(bundle.read(entry))\\n    archive.unlink()\\nos.chdir(project)\\nassets = [\'weights/efficientnetv2_s_best.pth\', \'weights/convnext_tiny_best.pth\',\\n          \'weights/swin_tiny_best.pth\', \'weights/maxvit_tiny_best.pth\',\\n          \'weights/densenet121_best.pth\', \'datasets/hackathon_dataset.zip\']\\nexpected_assets=json.loads(\'{\\"weights/efficientnetv2_s_best.pth\\": {\\"sha256\\": \\"3e5a3858cd40f22bfb885fd99aaedeff1c049ad720a49755e5e8d15a406d5b6e\\", \\"bytes\\": 81617723}, \\"weights/convnext_tiny_best.pth\\": {\\"sha256\\": \\"6165cde373bea423bc3da6e4d24d85f0b13bc44ee0bfcfff9b9f96dedfe18d3e\\", \\"bytes\\": 111355879}, \\"weights/swin_tiny_best.pth\\": {\\"sha256\\": \\"c45de824da9aa780f67b3ca597ed7a9352d6b957ce7cd825cb1f3b29019d02f3\\", \\"bytes\\": 110149322}, \\"weights/maxvit_tiny_best.pth\\": {\\"sha256\\": \\"10baf3a90f3a717aef75067ab87358c0c07717e827a5c3f2684c76082c56d083\\", \\"bytes\\": 122038277}, \\"weights/densenet121_best.pth\\": {\\"sha256\\": \\"683de13e7980c6e1365f877ee58ca81ed43f8e1ed87d0281f2f51f21d19bdc9e\\", \\"bytes\\": 28439894}, \\"datasets/hackathon_dataset.zip\\": {\\"sha256\\": \\"3fa8ff24720524a182a4cc449bb56f2ee48a588d04e442a8c3cd93a7089dc08e\\", \\"bytes\\": 133274437}}\')\\nfor relative in assets:\\n    path = project/relative\\n    text = path.read_text(encoding=\'utf-8\') if path.stat().st_size < 1024 else \'\'\\n    if text.startswith(\'version https://git-lfs.github.com/spec/v1\'):\\n        expected_hash = next(line.split(\'sha256:\')[1] for line in text.splitlines() if line.startswith(\'oid \'))\\n        expected_size = int(next(line.split()[1] for line in text.splitlines() if line.startswith(\'size \')))\\n        partial = path.with_suffix(path.suffix+\'.download\')\\n        print(\'[DOWNLOAD]\', relative, f\'({expected_size/1048576:.1f} MiB)\', flush=True)\\n        digest = hashlib.sha256()\\n        with urllib.request.urlopen(\'https://media.githubusercontent.com/media/snehav-Unique/SMPS_MODEL/\'+source_revision+\'/\'+relative, timeout=120) as response, partial.open(\'wb\') as output:\\n            while True:\\n                chunk = response.read(4*1024*1024)\\n                if not chunk: break\\n                output.write(chunk)\\n                digest.update(chunk)\\n        assert partial.stat().st_size == expected_size, \'Asset size mismatch: \'+relative\\n        assert digest.hexdigest() == expected_hash, \'Asset hash mismatch: \'+relative\\n        partial.replace(path)\\n        print(\'[VERIFIED]\', relative, flush=True)\\n    expected=expected_assets[relative]\\n    assert path.stat().st_size == expected[\'bytes\'], \'Cached asset size mismatch: \'+relative\\n    actual=hashlib.sha256()\\n    with path.open(\'rb\') as stream:\\n        for chunk in iter(lambda:stream.read(4*1024*1024),b\'\'):\\n            actual.update(chunk)\\n    assert actual.hexdigest() == expected[\'sha256\'], \'Cached asset hash mismatch: \'+relative\\n    print(\'[VERIFIED ASSET]\', relative, flush=True)\\nprint(\'[SETUP COMPLETE] Official dataset and all five checkpoints are available.\', flush=True)\\n", "scripts/robustness_test.py": "\\"\\"\\"Reproducible, resumable 200-image paired robustness evaluation.\\nRun: python SMPS/scripts/robustness_test.py (from workspace root).\\n\\"\\"\\"\\n\\nimport sys as _sys\\nfrom pathlib import Path as _Path\\n_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))\\n\\nimport hashlib\\nimport os\\nimport io\\nimport json\\nimport platform\\nimport time\\nimport zipfile\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\n\\nimport cv2\\nimport numpy as np\\nimport pandas as pd\\nimport torch\\nimport timm\\nfrom PIL import Image\\nfrom sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,\\n                             recall_score, f1_score, confusion_matrix)\\n\\nROOT = Path(__file__).resolve().parent.parent\\nOUT = ROOT / \'outputs\' / \'robustness\'\\nOUT.mkdir(parents=True, exist_ok=True)\\nSEED = 42\\nCONDITIONS = [\'clean\', \'jpeg_q50\', \'jpeg_q70\', \'blur_sigma1.5\', \'noise_sigma15\']\\nMODELS = {\\n    \'efficientnetv2_s\': (\'tf_efficientnetv2_s\', \'weights/efficientnetv2_s_best.pth\', .35),\\n    \'convnext_tiny\': (\'convnext_tiny\', \'weights/convnext_tiny_best.pth\', .30),\\n    \'swin_tiny\': (\'swin_tiny_patch4_window7_224\', \'weights/swin_tiny_best.pth\', .20),\\n    \'densenet121\': (\'densenet121\', \'weights/densenet121_best.pth\', .15),\\n}\\n\\ndef digest(path):\\n    h = hashlib.sha256()\\n    with open(path, \'rb\') as f:\\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\\n            h.update(chunk)\\n    return h.hexdigest()\\n\\ndef atomic_json(path, obj):\\n    tmp = path.with_suffix(\'.tmp\')\\n    tmp.write_text(json.dumps(obj, indent=2), encoding=\'utf-8\')\\n    tmp.replace(path)\\n\\ndef checkpoint(step, details):\\n    event = {\'utc\': datetime.now(timezone.utc).isoformat(), \'step\': step, \'details\': details}\\n    with (OUT / \'progress.jsonl\').open(\'a\', encoding=\'utf-8\') as f:\\n        f.write(json.dumps(event) + \'\\\\n\')\\n    atomic_json(OUT / \'checkpoint.json\', event)\\n    print(step, details, flush=True)\\n\\ndef prepare():\\n    archive = ROOT / \'datasets\' / \'hackathon_dataset.zip\'\\n    extraction = ROOT / \'datasets\' / \'extracted\'\\n    override = os.environ.get(\'SMPS_DATASET_ROOT\')\\n    existing = list(extraction.rglob(\'train_metadata.csv\')) if extraction.exists() else []\\n    if override:\\n        dataset = Path(override)\\n    elif existing:\\n        assert len(existing) == 1, \'Set SMPS_DATASET_ROOT for ambiguous extractions\'\\n        dataset = existing[0].parent\\n    else:\\n        extraction.mkdir(parents=True, exist_ok=True)\\n        with zipfile.ZipFile(archive) as z:\\n            for entry in z.infolist():\\n                target = (extraction / entry.filename).resolve()\\n                assert target.is_relative_to(extraction.resolve())\\n                if not entry.is_dir():\\n                    target.parent.mkdir(parents=True, exist_ok=True)\\n                    target.write_bytes(z.read(entry))\\n        metadata = list(extraction.rglob(\'train_metadata.csv\'))\\n        assert len(metadata) == 1, \'Expected one official dataset root\'\\n        dataset = metadata[0].parent\\n    val = pd.read_csv(ROOT / \'data/val_split.csv\')\\n    train = pd.read_csv(ROOT / \'data/train_split.csv\')\\n    assert val.image_id.is_unique and train.image_id.is_unique\\n    assert set(val.label) == {0, 1}\\n    assert not set(val.image_id) & set(train.image_id)\\n    metadata = pd.read_csv(dataset / \'train_metadata.csv\').set_index(\'image_id\')\\n    for row in val.itertuples():\\n        assert int(metadata.loc[row.image_id, \'label\']) == row.label\\n        image_path = dataset / \'train\' / (\'fake\' if row.label else \'real\') / (row.image_id + \'.jpg\')\\n        assert image_path.exists(), f\'Missing validation image: {image_path}\'\\n    sample = pd.concat([val[val.label == c].sort_values(\'image_id\').sample(n=100, random_state=SEED)\\n                        for c in [0, 1]]).sort_values(\'image_id\').reset_index(drop=True)\\n    sample.to_csv(OUT / \'sample_manifest.csv\', index=False)\\n    checkpoint(\'01_dataset_and_sample_verified\', {\'validation\': len(val), \'train\': len(train),\\n               \'sample_real\': 100, \'sample_fake\': 100, \'train_val_overlap\': 0})\\n    return dataset, sample\\n\\ndef perturb(a, condition, image_id):\\n    if condition.startswith(\'jpeg\'):\\n        b = io.BytesIO()\\n        Image.fromarray(a).save(b, format=\'JPEG\', quality=int(condition[-2:]), subsampling=2)\\n        return np.asarray(Image.open(io.BytesIO(b.getvalue())).convert(\'RGB\'))\\n    if condition.startswith(\'blur\'):\\n        return cv2.GaussianBlur(a, (0, 0), sigmaX=1.5, sigmaY=1.5, borderType=cv2.BORDER_REFLECT_101)\\n    if condition.startswith(\'noise\'):\\n        seed = int(hashlib.sha256(f\'{SEED}:{image_id}\'.encode()).hexdigest()[:16], 16)\\n        return np.clip(a.astype(np.float32) + np.random.default_rng(seed).normal(0, 15, a.shape), 0, 255).astype(np.uint8)\\n    return a\\n\\ndef inputs(dataset, sample):\\n    arrays = []\\n    for row in sample.itertuples():\\n        p = dataset / \'train\' / (\'fake\' if row.label else \'real\') / (row.image_id + \'.jpg\')\\n        a = np.asarray(Image.open(p).convert(\'RGB\'))\\n        assert a.shape == (224, 224, 3), (p, a.shape)\\n        arrays.append(a)\\n    result = {}\\n    for c in CONDITIONS:\\n        a = np.stack([perturb(a, c, row.image_id) for a, row in zip(arrays, sample.itertuples())])\\n        # Images are already 224x224; ImageNet normalization matches notebook validation.\\n        result[c] = ((torch.from_numpy(a.copy()).permute(0, 3, 1, 2).float() / 255\\n                      - torch.tensor([.485, .456, .406])[None, :, None, None])\\n                     / torch.tensor([.229, .224, .225])[None, :, None, None])\\n        Image.fromarray(a[0]).save(OUT / f\'example_{c}.png\')\\n    checkpoint(\'02_perturbations_prepared\', {\'conditions\': CONDITIONS, \'seed\': SEED})\\n    return result\\n\\ndef infer(sample, tensors, device):\\n    for name, (architecture, weights, _) in MODELS.items():\\n        model = timm.create_model(architecture, pretrained=False, num_classes=1)\\n        state = torch.load(ROOT / weights, map_location=\'cpu\', weights_only=True)\\n        if \'state_dict\' in state: state = state[\'state_dict\']\\n        elif \'model\' in state: state = state[\'model\']\\n        model.load_state_dict(state, strict=True)\\n        model.to(device).eval()\\n        checkpoint(\'03_model_loaded\', {\'model\': name, \'strict_load\': True})\\n        for c in CONDITIONS:\\n            path = OUT / f\'predictions_{name}_{c}.csv\'\\n            previous = pd.read_csv(path) if path.exists() else pd.DataFrame(columns=[\'image_id\', \'true_label\', \'confidence\'])\\n            assert previous.image_id.tolist() == sample.image_id.tolist()[:len(previous)]\\n            rows = previous.to_dict(\'records\')\\n            started = time.perf_counter()\\n            with torch.inference_mode():\\n                for start in range(len(rows), len(sample), 8):\\n                    x = tensors[c][start:start+8].to(device)\\n                    # Two-view original + horizontal-flip TTA, as used in saved test predictions.\\n                    p = (torch.sigmoid(model(x)).flatten() + torch.sigmoid(model(x.flip(-1))).flatten()) / 2\\n                    probabilities = p.cpu().numpy()\\n                    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()\\n                    for row, prob in zip(sample.iloc[start:start+8].itertuples(), probabilities):\\n                        rows.append({\'image_id\': row.image_id, \'true_label\': row.label, \'confidence\': float(prob)})\\n                    tmp = path.with_suffix(\'.tmp\')\\n                    pd.DataFrame(rows).to_csv(tmp, index=False)\\n                    tmp.replace(path)\\n                    checkpoint(\'04_batch_saved\', {\'model\': name, \'condition\': c, \'images\': len(rows), \'total\': len(sample)})\\n            checkpoint(\'04_condition_complete\', {\'model\': name, \'condition\': c, \'seconds_this_run\': time.perf_counter()-started})\\n        del model\\n\\ndef summarize(sample):\\n    import matplotlib\\n    matplotlib.use(\'Agg\')\\n    import matplotlib.pyplot as plt\\n    y = sample.label.to_numpy()\\n    scores = {}\\n    for c in CONDITIONS:\\n        scores[c] = np.zeros(len(y))\\n        for name, (_, _, weight) in MODELS.items():\\n            df = pd.read_csv(OUT / f\'predictions_{name}_{c}.csv\')\\n            assert df.image_id.tolist() == sample.image_id.tolist() and df.true_label.tolist() == y.tolist()\\n            scores[c] += weight * df.confidence.to_numpy()\\n        df = sample.rename(columns={\'label\': \'true_label\'}).copy()\\n        df[\'confidence\'] = scores[c]\\n        df[\'predicted_label\'] = (scores[c] >= .5).astype(int)\\n        df.to_csv(OUT / f\'predictions_ensemble_{c}.csv\', index=False)\\n    rng = np.random.default_rng(SEED)\\n    boot = np.stack([np.concatenate([rng.choice(np.where(y == k)[0], 100, replace=True) for k in [0, 1]]) for _ in range(2000)])\\n    clean_boot = np.array([roc_auc_score(y[i], scores[\'clean\'][i]) for i in boot])\\n    clean_auc = roc_auc_score(y, scores[\'clean\'])\\n    rows = []\\n    for c, p in scores.items():\\n        pred = p >= .5\\n        tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\\n        auc = roc_auc_score(y, p)\\n        auc_boot = np.array([roc_auc_score(y[i], p[i]) for i in boot])\\n        ci = np.quantile(auc_boot, [.025, .975])\\n        dci = np.quantile(clean_boot - auc_boot, [.025, .975])\\n        rows.append(dict(condition=c, n=len(y), auc=auc, auc_ci_low=ci[0], auc_ci_high=ci[1],\\n            auc_drop=clean_auc-auc, drop_ci_low=dci[0], drop_ci_high=dci[1],\\n            relative_auc_drop_pct=100*(clean_auc-auc)/clean_auc,\\n            accuracy=accuracy_score(y,pred), precision=precision_score(y,pred,zero_division=0),\\n            recall=recall_score(y,pred), f1=f1_score(y,pred), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp),\\n            observed_drop_below_002=bool(clean_auc-auc < .02)))\\n    stats = pd.DataFrame(rows)\\n    stats.to_csv(OUT / \'statistics.csv\', index=False)\\n    fig, ax = plt.subplots(figsize=(9, 5))\\n    ax.bar(stats.condition, stats.auc, color=[\'#2563eb\']+[\'#0891b2\']*4)\\n    ax.errorbar(range(5), stats.auc, yerr=[stats.auc-stats.auc_ci_low, stats.auc_ci_high-stats.auc], fmt=\'none\', color=\'black\', capsize=4)\\n    ax.axhline(clean_auc-.02, color=\'#dc2626\', linestyle=\'--\', label=\'Clean AUC minus 0.02\')\\n    ax.set(ylim=(0,1), ylabel=\'ROC AUC\', title=\'Paired robustness test: 200 validation images (95% bootstrap CI)\')\\n    ax.legend(); fig.tight_layout(); fig.savefig(OUT / \'auc_comparison.png\', dpi=180); plt.close(fig)\\n    table = \'| Condition | AUC (95% CI) | AUC drop | Accuracy | Precision | Recall | F1 | TN / FP / FN / TP | <0.02 drop? |\\\\n|---|---|---|---|---|---|---|---|---|\\\\n\'\\n    for r in stats.itertuples():\\n        table += f\'| {r.condition} | {r.auc:.4f} ({r.auc_ci_low:.4f}\\u2013{r.auc_ci_high:.4f}) | {r.auc_drop:.4f} | {r.accuracy:.3f} | {r.precision:.3f} | {r.recall:.3f} | {r.f1:.3f} | {r.tn} / {r.fp} / {r.fn} / {r.tp} | {\\"Yes\\" if r.observed_drop_below_002 else \\"No\\"} |\\\\n\'\\n    worst = stats.iloc[1:].sort_values(\'auc_drop\', ascending=False).iloc[0]\\n    report = f\'\'\'# Deepfake detector robustness report\\n\\n## Findings\\n\\nClean ensemble AUC: **{clean_auc:.4f}**. Largest observed degradation: **{worst[\'condition\']}**, with AUC **{worst[\'auc\']:.4f}** and absolute drop **{worst[\'auc_drop\']:.4f}**.\\n{int(stats.iloc[1:].observed_drop_below_002.sum())} of 4 corruption conditions meet the observed absolute AUC-drop target of less than 0.02.\\n\\n{table}\\n\\n## Method\\n\\n200 images (100 real, 100 fake), sampled without replacement from the existing 1,400-image validation split; seed 42. All validation IDs were checked against archive metadata and checked for no ID overlap with the 5,600-image training split. The same images are paired across all conditions. No models were trained and no thresholds or ensemble weights were tuned on these results.\\n\\nFour-model probability ensemble: EfficientNetV2-S 0.35, ConvNeXt-Tiny 0.30, Swin-Tiny 0.20, DenseNet-121 0.15. Strict checkpoint loading; evaluation mode; FP32; original and horizontal-flip probability averaging; RGB 224x224; ImageNet normalization. Classification threshold 0.5; positive class is fake. Environment and SHA-256 weight hashes are in run_manifest.json.\\n\\nEach corruption is independently applied to the decoded original: JPEG quality 50/70 with 4:2:0 subsampling; OpenCV Gaussian blur sigma=1.5 with automatic kernel and reflect-101 borders; independent Gaussian pixel noise sigma=15 on the 0\\u2013255 scale, clipped and cast to uint8. Noise is deterministic per image. Corruptions are not stacked.\\n\\n95% percentile intervals use 2,000 stratified bootstrap resamples, paired between clean and degraded images. statistics.csv includes confidence intervals for the AUC drops and relative percentage drops. The target is interpreted as an **absolute drop below 0.02**, not a relative 2% decrease.\\n\\n## Interpretation and limitations\\n\\nThese are measured validation-sample results, not hidden-test guarantees. A 200-image sample has sampling uncertainty; meeting the observed target does not establish that the true drop is below 0.02. Models were selected using this validation split, so these are not independent final-test estimates. ID separation does not establish identity/source separation. Noise uses one fixed realization per image. No T4 speed or VRAM score is claimed from this CPU run. The handover\'s projected ensemble AUC and robustness claims were not treated as measured results.\\n\\n## Files and resuming\\n\\n- statistics.csv: full metrics and uncertainty estimates.\\n- predictions_*.csv: all model and ensemble per-image probabilities.\\n- sample_manifest.csv: exact selected IDs and ground truth.\\n- auc_comparison.png: AUC comparison with confidence intervals.\\n- example_*.png: one image under each condition.\\n- progress.jsonl and checkpoint.json: append-only progress history and latest checkpoint.\\n- run_manifest.json: reproducibility settings and hashes.\\n\\nResume or regenerate the report by running `python SMPS/scripts/robustness_test.py` from the workspace root. Completed prediction batches are reused only when the run signature matches.\\n\'\'\'\\n    (OUT / \'REPORT.md\').write_text(report, encoding=\'utf-8\')\\n    checkpoint(\'05_statistics_and_report_complete\', {\'clean_auc\': clean_auc, \'worst_condition\': worst[\'condition\'], \'worst_auc_drop\': float(worst[\'auc_drop\'])})\\n    print(stats.to_string(index=False), flush=True)\\n\\ndef main():\\n    torch.set_num_threads(4)\\n    torch.manual_seed(SEED)\\n    dataset, sample = prepare()\\n    device = \'cuda\' if torch.cuda.is_available() else \'cpu\'\\n    manifest = {\'protocol_version\': 1, \'seed\': SEED, \'device\': device, \'precision\': \'float32\', \'tta\': \'original+horizontal_flip\',\\n        \'batch_size\': 8, \'threads\': 4, \'conditions\': CONDITIONS, \'threshold\': .5,\\n        \'python\': platform.python_version(), \'torch\': torch.__version__, \'timm\': timm.__version__,\\n        \'numpy\': np.__version__, \'opencv\': cv2.__version__, \'pillow\': Image.__version__,\\n        \'sample_sha256\': digest(OUT / \'sample_manifest.csv\'), \'script_sha256\': digest(Path(__file__)),\\n        \'dataset_path\': str(dataset), \'sample_image_sha256\': {r.image_id: digest(dataset / \'train\' / (\'fake\' if r.label else \'real\') / (r.image_id+\'.jpg\')) for r in sample.itertuples()},\\n        \'models\': {name: {\'architecture\': a, \'path\': p, \'weight\': w, \'sha256\': digest(ROOT / p)} for name, (a,p,w) in MODELS.items()}}\\n    manifest_path = OUT / \'run_manifest.json\'\\n    if manifest_path.exists():\\n        assert json.loads(manifest_path.read_text(encoding=\'utf-8\')) == manifest, \'Run signature changed; use a new output folder.\'\\n    atomic_json(manifest_path, manifest)\\n    checkpoint(\'00_environment_recorded\', {\'device\': device, \'torch\': torch.__version__, \'timm\': timm.__version__})\\n    tensors = inputs(dataset, sample)\\n    infer(sample, tensors, device)\\n    summarize(sample)\\n\\nif __name__ == \'__main__\':\\n    try:\\n        main()\\n    except Exception as e:\\n        checkpoint(\'ERROR\', {\'type\': type(e).__name__, \'message\': str(e)})\\n        raise\\n", "scripts/render_dashboard.py": "\\"\\"\\"Render the presentation dashboard from existing measured charts; no inference.\\"\\"\\"\\n\\nimport sys as _sys\\nfrom pathlib import Path as _Path\\n_sys.path.insert(0, str(_Path(__file__).resolve().parent.parent))\\n\\nimport base64\\nimport html\\nfrom pathlib import Path\\n\\nROOT = Path(__file__).resolve().parent.parent\\nCHARTS = ROOT/\'outputs/final_verification\'\\n\\n\\ndef data_image(path):\\n    return \'data:image/png;base64,\' + base64.b64encode(path.read_bytes()).decode()\\n\\n\\ncharts = [\\n    (\'final_roc\', \'performance\', \'01 / PREDICTIVE PERFORMANCE\', \'A closer look at every decision\', \'ROC curve and confusion matrix \\u00b7 200 balanced validation images \\u00b7 Threshold 0.5\', True),\\n    (\'model_auc\', \'architectures\', \'02 / ARCHITECTURE COMPARISON\', \'Five perspectives. One ensemble.\', \'Recorded individual peak validation AUC\', False),\\n    (\'learning\', \'training\', \'03 / LEARNING DYNAMICS\', \'Learning versus generalization\', \'Saved training and validation histories\', False),\\n    (\'final_robustness\', \'robustness\', \'04 / ROBUSTNESS\', \'Performance under pressure\', \'Final five-model ensemble \\u00b7 200 validation images \\u00b7 95% bootstrap intervals\', True),\\n    (\'confidence\', \'confidence\', \'05 / PREDICTION DISTRIBUTION\', \'The shape of confidence\', \'3,000 unlabelled test images \\u00b7 Predicted probability of Fake\', False),\\n    (\'final_timing\', \'efficiency\', \'06 / EFFICIENCY\', \'Measured on Tesla T4\', \'Final five-model FP32 pipeline \\u00b7 End-to-end process wall time\', False),\\n]\\ncards = []\\nfor name, anchor, label, title, caption, wide in charts:\\n    path = CHARTS/(name+\'.png\')\\n    if not path.exists():\\n        continue\\n    cards.append(f\'\'\'<article id=\\"{anchor}\\" class=\\"chart {\'wide\' if wide else \'\'}\\">\\n      <div class=\\"card-heading\\"><span class=\\"eyebrow\\">{label}</span><span class=\\"corner\\" aria-hidden=\\"true\\">\\u2197</span></div>\\n      <h2>{title}</h2><p class=\\"caption\\">{caption}</p>\\n      <button class=\\"visual\\" aria-label=\\"Enlarge {html.escape(title)}\\"><img src=\\"{data_image(path)}\\" alt=\\"{html.escape(title)}\\"></button>\\n    </article>\'\'\')\\n\\ngallery = \'\'.join(f\'<button class=\\"visual gradcam\\" aria-label=\\"Enlarge Grad-CAM example {i+1}\\"><img src=\\"{data_image(p)}\\" alt=\\"Grad-CAM example {i+1}: original image, attention heatmap and overlay\\"></button>\'\\n                  for i,p in enumerate(sorted((ROOT/\'outputs/gradcam\').glob(\'*.png\'))[:4]))\\npage = \'\'\'<!doctype html>\\n<html lang=\\"en\\"><head><meta charset=\\"utf-8\\"><meta name=\\"viewport\\" content=\\"width=device-width,initial-scale=1\\">\\n<title>SMPS \\u2014 Deepfake Detection</title>\\n<style>\\n:root{--ink:#153b35;--muted:#6a7972;--paper:#f5f5ef;--line:#dee5dc;--accent:#b8d998}*{box-sizing:border-box}html{scroll-behavior:smooth;scroll-padding-top:90px}body{margin:0;background:var(--paper);color:var(--ink);font-family:\'Segoe UI\',Arial,sans-serif;-webkit-font-smoothing:antialiased}a{color:inherit;text-decoration:none}header{background:#143b35;color:#fff;position:relative;overflow:hidden}header:after{content:\'\';position:absolute;width:560px;height:560px;border:1px solid #ffffff16;border-radius:50%;right:-70px;top:-180px;box-shadow:0 0 0 75px #ffffff04,0 0 0 150px #ffffff03;pointer-events:none}.topbar{max-width:1320px;margin:auto;padding:26px 44px;display:flex;justify-content:space-between;align-items:center;position:relative;z-index:1}.brand{font-size:22px;letter-spacing:.15em;font-weight:750;display:flex;align-items:center;gap:12px}.brand:before{content:\'\';width:13px;height:13px;background:var(--accent);border-radius:50%;box-shadow:0 0 0 5px #ffffff0b}.event{font-size:11px;letter-spacing:.17em;color:#bdcfc6}.hero{max-width:1320px;margin:auto;padding:46px 44px 66px;position:relative;z-index:1}.hero .eyebrow{color:#c5dfa9}.hero h1{font:normal clamp(42px,5.5vw,74px)/1.06 Georgia,serif;letter-spacing:-.035em;margin:20px 0 22px;max-width:800px}.hero h1 em{font-weight:normal;color:#c4dfa8}.hero p{margin:0;font-size:16px;color:#c8d6cc;letter-spacing:.015em}.hero-bottom{display:flex;justify-content:space-between;align-items:center;gap:20px}.signature{border:1px solid #ffffff30;border-radius:40px;padding:10px 18px;font-size:11px;letter-spacing:.13em;color:#cce0cf;white-space:nowrap}.nav-wrap{background:#f5f5eff2;backdrop-filter:blur(18px);position:sticky;top:0;z-index:5;border-bottom:1px solid var(--line)}nav{max-width:1320px;margin:auto;padding:0 44px;display:flex;gap:34px;overflow:auto}nav a{display:block;padding:21px 0;font-size:12px;letter-spacing:.06em;font-weight:600;white-space:nowrap;border-bottom:2px solid transparent}nav a:hover{border-color:var(--ink)}main{max-width:1320px;margin:auto;padding:44px}.section-heading{display:flex;justify-content:space-between;align-items:baseline;margin:0 0 25px}.section-heading h2{font:normal 28px Georgia,serif;margin:0}.section-heading span{font-size:10px;letter-spacing:.12em;color:var(--muted)}.grid{display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:24px}.chart{min-width:0;border:1px solid var(--line);background:#fff;border-radius:14px;padding:28px 28px 16px;box-shadow:0 5px 16px #153b3503;transition:transform .2s,box-shadow .2s}.chart:hover{transform:translateY(-3px);box-shadow:0 13px 28px #153b3509}.wide{grid-column:1/-1}.card-heading{display:flex;justify-content:space-between;align-items:center}.eyebrow{font-size:10px;font-weight:650;letter-spacing:.15em;color:#607c6c}.corner{color:#718676;font-size:20px}.chart h2{font:normal clamp(22px,2.2vw,30px)/1.18 Georgia,serif;letter-spacing:-.025em;margin:17px 0 9px}.caption{font-size:12px;line-height:1.6;color:var(--muted);margin:0 0 22px}.visual{display:block;background:#fff;border:0;padding:0;width:100%;cursor:zoom-in;text-align:inherit;border-radius:8px;overflow:hidden}.visual img{display:block;width:100%;height:auto}.wide .visual{max-width:1030px;margin:auto}.attention{margin-top:56px;background:#e7ede2;border:1px solid #d8e1d2;border-radius:16px;padding:32px}.attention h2{font:normal 32px Georgia,serif;margin:14px 0 12px;letter-spacing:-.025em}.attention p{font-size:12px;color:#607367;margin:0 0 25px}.gallery{display:grid;grid-template-columns:1fr 1fr;gap:18px}.gradcam{border:8px solid white;box-shadow:0 4px 14px #153b3508}.footnote{font-size:11px;line-height:1.8;color:#748178;max-width:880px;margin:27px 0 0}footer{border-top:1px solid var(--line);margin-top:44px;padding:25px 0 0;display:flex;justify-content:space-between;font-size:10px;letter-spacing:.11em;color:#6f7d72}.lightbox{position:fixed;inset:0;z-index:20;background:#0b211bef;display:none;align-items:center;justify-content:center;padding:65px 28px 28px;backdrop-filter:blur(12px)}.lightbox.open{display:flex}.lightbox img{max-width:100%;max-height:100%;object-fit:contain;border-radius:12px;background:white}.close{position:absolute;right:28px;top:18px;color:white;background:#ffffff15;border:1px solid #ffffff30;border-radius:50%;width:38px;height:38px;font-size:25px;cursor:pointer}.visual:focus-visible,a:focus-visible,.close:focus-visible{outline:3px solid #79aa68;outline-offset:5px}@media(max-width:760px){.topbar{padding:24px}.event{font-size:9px;max-width:150px;text-align:right;line-height:1.8}.hero{padding:30px 24px 42px}.hero-bottom{display:block}.signature{display:inline-block;margin-top:24px}nav{padding:0 24px;gap:24px}main{padding:30px 18px}.grid,.gallery{grid-template-columns:1fr}.chart{padding:23px 18px 14px}.wide{grid-column:auto}.section-heading span{display:none}.attention{padding:24px 18px}.attention h2{font-size:28px}footer{gap:15px;line-height:1.8}}@media(prefers-reduced-motion:reduce){html{scroll-behavior:auto}.chart{transition:none}}@media print{.nav-wrap,.lightbox{display:none}header{print-color-adjust:exact}main{padding:20px}.chart,.attention{break-inside:avoid;box-shadow:none}.grid{display:block}.chart{margin-bottom:20px}}\\n</style></head><body>\\n<header><div class=\\"topbar\\"><a class=\\"brand\\" href=\\"#\\">SMPS</a><span class=\\"event\\">IIIT BANGALORE \\u00b7 DEEPFAKE ML HACKATHON</span></div>\\n<div class=\\"hero\\"><span class=\\"eyebrow\\">DEEPFAKE DETECTION / EXPERIMENTAL RESULTS</span><h1>Beyond the pixels.<br><em>Inside the evidence.</em></h1><div class=\\"hero-bottom\\"><p>A visual study of performance, resilience, and attention.</p><span class=\\"signature\\">FIVE-MODEL ENSEMBLE</span></div></div></header>\\n<div class=\\"nav-wrap\\"><nav aria-label=\\"Results sections\\"><a href=\\"#performance\\">Performance</a><a href=\\"#architectures\\">Architectures</a><a href=\\"#training\\">Training</a><a href=\\"#robustness\\">Robustness</a><a href=\\"#efficiency\\">Efficiency</a><a href=\\"#explainability\\">Explainability</a></nav></div>\\n<main><div class=\\"section-heading\\"><h2>The results, in focus.</h2><span>TEAM SMPS / NOVEMBER 2026</span></div><div class=\\"grid\\">\'\'\'\\npage += \'\'.join(cards)\\npage += \'<div class=\\"attention wide\\" id=\\"explainability\\"><span class=\\"eyebrow\\">07 / VISUAL EXPLAINABILITY</span><h2>Where attention lands.</h2><p>Original image \\u00b7 Grad-CAM heatmap \\u00b7 Attention overlay</p><div class=\\"gallery\\">\'+gallery+\'</div></div></div>\'\\npage += \'\'\'<p class=\\"footnote\\">Validation and robustness charts use a fixed, balanced sample of 200 held-out images. Error bars represent 95% bootstrap intervals. Test confidence distributions do not measure test accuracy. Grad-CAM shows attribution, not causal proof.</p><footer><span>SMPS / DEEPFAKE DETECTION</span><span>MEASURED RESULTS \\u00b7 PRIVATE TEST SCORE UNKNOWN</span></footer></main>\\n<div class=\\"lightbox\\" role=\\"dialog\\" aria-modal=\\"true\\" aria-label=\\"Expanded chart\\"><button class=\\"close\\" aria-label=\\"Close expanded chart\\">\\u00d7</button><img alt=\\"\\"></div>\\n<script>const box=document.querySelector(\'.lightbox\'),large=box.querySelector(\'img\'),close=box.querySelector(\'.close\');let previous;document.querySelectorAll(\'.visual\').forEach(button=>button.addEventListener(\'click\',()=>{previous=button;large.src=button.querySelector(\'img\').src;large.alt=button.querySelector(\'img\').alt;box.classList.add(\'open\');document.body.style.overflow=\'hidden\';close.focus()}));function dismiss(){box.classList.remove(\'open\');document.body.style.overflow=\'\';previous?.focus()}close.addEventListener(\'click\',dismiss);box.addEventListener(\'click\',e=>{if(e.target===box)dismiss()});document.addEventListener(\'keydown\',e=>{if(e.key===\'Escape\')dismiss();if(e.key===\'Tab\'&&box.classList.contains(\'open\')){e.preventDefault();close.focus()}});</script></body></html>\'\'\'\\nintro_head = \'\'\'<script>document.documentElement.classList.add(\'intro-pending\');if(\'scrollRestoration\' in history)history.scrollRestoration=\'manual\';</script>\\n<style>\\nhtml.intro-pending{overflow:hidden;background:#f5f5ef}html.intro-pending body{visibility:hidden}\\nhtml.intro-pending .topbar,html.intro-pending .hero>.eyebrow,html.intro-pending .hero h1,html.intro-pending .hero-bottom,html.intro-pending .nav-wrap,html.intro-pending .section-heading,html.intro-pending .chart,html.intro-pending .attention,html.intro-pending .footnote,html.intro-pending footer{opacity:0}\\n.assembling{opacity:0;transform:translate3d(var(--from-x,0),var(--from-y,0),0) scale(.97);transition:opacity .8s ease var(--entrance-delay,0s),transform 1.05s cubic-bezier(.16,1,.3,1) var(--entrance-delay,0s)}\\n.assembling.assembled{opacity:1;transform:translate3d(0,0,0) scale(1)}\\n@media(prefers-reduced-motion:reduce){.assembling{transform:none;transition:opacity .25s ease}}\\n@media print{html.intro-pending body{visibility:visible}.assembling{opacity:1!important;transform:none!important}}\\n</style>\'\'\'\\nintro_script = \'\'\'<script>\\n(()=>{\\nconst selectors=[\'.topbar\',\'.hero>.eyebrow\',\'.hero h1\',\'.hero-bottom\',\'.nav-wrap\',\'.section-heading\',\'.chart\',\'.attention\',\'.footnote\',\'footer\'];\\nconst elements=[...document.querySelectorAll(selectors.join(\',\'))];\\nconst directions=[[0,-55],[-100,0],[-150,15],[140,0],[0,-40],[-90,0],[-130,35],[130,35],[-130,35],[130,35],[-130,35],[130,35],[0,90],[0,40],[0,35]];\\nelements.forEach((element,index)=>{const [x,y]=directions[index%directions.length];element.classList.add(\'assembling\');element.style.setProperty(\'--from-x\',x+\'px\');element.style.setProperty(\'--from-y\',y+\'px\');element.style.setProperty(\'--entrance-delay\',Math.min(index*.085,1.05)+\'s\')});\\nsetTimeout(()=>{\\nwindow.scrollTo(0,0);\\ndocument.documentElement.classList.remove(\'intro-pending\');\\nrequestAnimationFrame(()=>requestAnimationFrame(()=>elements.forEach(element=>element.classList.add(\'assembled\'))));\\nsetTimeout(()=>elements.forEach(element=>{element.classList.remove(\'assembling\',\'assembled\');element.style.removeProperty(\'--entrance-delay\')}),2400);\\n},Math.max(0,2000-performance.now()));\\n})();\\n</script>\'\'\'\\npage = page.replace(\'<style>\', intro_head+\'<style>\', 1)\\npage = page.replace(\'</body>\', intro_script+\'</body>\')\\n(ROOT/\'presentation/results.html\').write_text(page, encoding=\'utf-8\')\\nprint(\'Built presentation dashboard with measured charts only.\')\\n", "scripts/generate_final_report.py": "\\"\\"\\"Build the four-page final report from recorded evidence; no model execution.\\"\\"\\"\\nfrom pathlib import Path\\nimport csv\\nimport json\\nfrom reportlab.lib import colors\\nfrom reportlab.lib.pagesizes import A4\\nfrom reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle\\nfrom reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, PageBreak\\n\\nROOT=Path(__file__).resolve().parent.parent\\nOUT=ROOT/\'docs/reports/final_technical_report.pdf\'\\n\\ndef main():\\n    evidence=json.loads((ROOT/\'outputs/final_verification/verification.json\').read_text(encoding=\'utf-8\'))\\n    config=json.loads((ROOT/\'weights/ensemble_config.json\').read_text(encoding=\'utf-8\'))\\n    with (ROOT/\'outputs/final_validation/statistics.csv\').open() as stream:\\n        stats=list(csv.DictReader(stream))\\n    styles=getSampleStyleSheet()\\n    styles.add(ParagraphStyle(name=\'BodyFinal\',fontName=\'Helvetica\',fontSize=10,leading=14,spaceAfter=9))\\n    story=[]\\n    def p(text,style=\'BodyFinal\'):story.append(Paragraph(text,styles[style]))\\n    def table(rows,widths):\\n        rows=[[Paragraph(str(x),styles[\'BodyFinal\']) for x in row] for row in rows]\\n        t=Table(rows,colWidths=widths,hAlign=\'LEFT\')\\n        t.setStyle(TableStyle([(\'BACKGROUND\',(0,0),(-1,0),colors.HexColor(\'#e3edf1\')),(\'VALIGN\',(0,0),(-1,-1),\'TOP\'),(\'BOTTOMPADDING\',(0,0),(-1,-1),7),(\'TOPPADDING\',(0,0),(-1,-1),7),(\'LINEBELOW\',(0,0),(-1,-1),0.3,colors.HexColor(\'#c7d4db\'))]))\\n        story.append(t);story.append(Spacer(1,10))\\n    p(\'Deepfake face classification\',\'Title\')\\n    p(\'Team SMPS | IIIT Bangalore Deepfake ML Hackathon | Final technical report\')\\n    p(\'1. Problem, dataset and evaluation design\',\'Heading1\')\\n    p(\'The task is binary classification of aligned 224 x 224 face images. Label 0 denotes Real and label 1 denotes Fake. The confidence column is the estimated probability of Fake, rather than confidence in whichever label was selected. The primary ranking metric in the participant package is AUC-ROC. Labels are produced at threshold 0.5; this threshold is not claimed to be optimal for every use case.\')\\n    p(\'The supplied archive contains 7,000 labelled training images: 3,478 Real and 3,522 Fake. It also contains 3,000 unlabelled official test images. Saved CSVs define a stratified 80/20 split of 5,600 training and 1,400 validation images, with seed 42. Only the supplied dataset may be used for fine-tuning. ImageNet initialization is permitted. Official test images are used for inference, not training or tuning.\')\\n    p(\'Final robustness statistics use a fixed, balanced subset of 200 held-out validation images, 100 per class. This subset is smaller than the full validation set and is not a second independent test set. Models and checkpoints were selected using validation information. Consequently, results must be interpreted as validation evidence, not an unbiased estimate of private-test performance.\')\\n    p(\'The final test prediction file contains exactly one row for each official test image ID. Its schema is image_id,label,confidence; image IDs omit the filename extension. Private test labels are unavailable. We do not report private-test AUC, official points, projected points, or a guaranteed ranking.\')\\n    p(\'Ethical use\',\'Heading2\')\\n    p(\'False positives can incorrectly flag genuine people and false negatives can miss manipulated media. A model output is supporting evidence for review, not a definitive finding of fabrication. Performance on these aligned face images does not establish performance on arbitrary videos, unseen demographic groups, full images, or new manipulation methods. No demographic fairness or calibration study is claimed.\')\\n    p(\'Evidence sources: datasets/participant_instructions/, data/train_split.csv, data/val_split.csv, outputs/final_validation/, and outputs/final_verification/. Earlier reports are preserved as historical artifacts and are superseded by this report for final claims.\')\\n    story.append(PageBreak())\\n    p(\'2. Architecture and inference\',\'Heading1\')\\n    table([[\'Model / timm architecture\',\'Blend\',\'Recorded peak AUC\']]+[[m[\'arch\'],f\\"{m[\'weight\']:.2f}\\",f\\"{m[\'val_auc\']:.4f}\\"] for m in config[\'models\']],[285,65,145])\\n    p(\'The ensemble combines convolutional and attention-based architectures with different feature-processing mechanisms: EfficientNetV2-S, ConvNeXt-Tiny, Swin-Tiny, MaxViT-Tiny and DenseNet-121. Each produces a single logit; sigmoid converts it to a Fake probability. Preprocessing resizes to 224 x 224, converts to a tensor, and applies ImageNet normalization.\')\\n    p(\'For each model, inference averages predictions on the original image and its horizontal flip. The final confidence is the weighted sum of these per-model probabilities. Blend coefficients are nonnegative and sum to one. They are fixed heuristic coefficients, not the result of a demonstrated optimal search. Diversity motivates the combination but does not establish that it improves on the strongest individual model.\')\\n    p(\'Individual peak AUC values above are recorded model-selection results, whereas final ensemble robustness uses the fixed 200-image subset. These samples and selection procedures differ; direct subtraction would not measure ensemble gain. No controlled ablation proving the benefit of all five models is claimed.\')\\n    p(\'The official CLI loads weights/ensemble_config.json. That file references five checkpoints in the same directory. ensemble_worker_control.py launches each model in a separate process, releasing model memory between workers. The verified path uses FP32 with original-plus-horizontal-flip TTA and batch size 64 by default. Keep the inference entry point and worker controller together.\')\\n    p(\'Reproduction command\',\'Heading2\')\\n    p(\'<font face=\\"Courier\\" size=\\"8\\">python inference_script.py --test_dir PATH/test/images<br/>--weights weights/ensemble_config.json --output fresh_predictions.csv</font>\')\\n    p(\'Run the command on one line from the repository root. Input paths are supplied explicitly. Download binary assets through Git LFS; pointer files are not usable checkpoints. Baseline, TensorFlow and ONNX files remain available, but they are not the final verified five-model PyTorch pipeline. Cross-framework numerical parity has not been established.\')\\n    story.append(PageBreak())\\n    p(\'3. Training, hyperparameters and challenges\',\'Heading1\')\\n    p(\'Architecture-specific notebooks under notebooks/training/ are the source of training settings. The generic src/train.py is an earlier baseline scaffold using BCEWithLogitsLoss, AdamW and cosine annealing; running it alone does not reproduce all five architecture-specific training runs. Install requirements-training.txt for that generic pipeline, which uses the Albumentations 1.x API.\')\\n    table([[\'Notebook family\',\'Recorded source settings\'],[\'ConvNeXt, DenseNet, EfficientNetV2-S, Swin\',\'Batch 32; learning rate 1e-4; seed 42; configured 25 epochs; focal gamma 1.5; default alpha 0.5; AdamW effective weight decay 1e-3; cosine warm restarts T_0=10, T_mult=2, eta_min=1e-6.\'],[\'MaxViT saved-results notebook\',\'Batch 32; learning rate 1e-4; seed 42; configured 20 epochs; focal alpha 0.5, gamma 1.5; AdamW weight decay 1e-4; cosine warm restarts T_0=5, T_mult=2, eta_min=1e-6.\']],[145,350])\\n    p(\'Configured epoch limits are not claims about the epoch that produced each saved checkpoint. Saved logs and notebook outputs retain actual histories. Some notebooks define a weight-decay variable but pass a different literal value to AdamW; the table reports the effective optimizer call. Settings are therefore not described as identical across architectures.\')\\n    p(\'Training includes geometric and photometric augmentation and compression/noise perturbations on supplied training images. Validation preprocessing is deterministic. Exact transform parameters belong to each notebook; a single universal augmentation recipe would misdescribe the retained experiments. Checkpoints are selected using validation AUC rather than assuming the final epoch is best.\')\\n    p(\'Hyperparameter selection and limitations\',\'Heading2\')\\n    p(\'The repository records chosen configurations and exploratory models rather than a complete systematic hyperparameter search. We do not claim that the learning rate, focal loss, blend coefficients or threshold are globally optimal. Strong training performance with plateauing validation performance motivates careful checkpoint selection and restraint in performance claims.\')\\n    p(\'EVA-02 Tiny is retained as an exploratory comparison, with recorded peak AUC 0.7126. Architecture and limited data are possible explanations for the result; the project does not isolate a causal failure mechanism. GPU memory and local CPU resource constraints motivated sequential worker inference. A previous precomputed ensemble CSV failed to match the current pipeline and is preserved separately as predictions_original.csv; the final CSV was generated by the actual FP32 pipeline and independently replayed.\')\\n    story.append(PageBreak())\\n    p(\'4. Final measurements, robustness and delivery\',\'Heading1\')\\n    run=evidence[\'runs\'][0]\\n    p(f\\"Recorded independent replay processed all 3,000 official test images on {evidence[\'device\']}. Maximum probability difference was {run[\'reproduction\'][\'max_probability_difference\']:.1f}, with {run[\'reproduction\'][\'label_mismatches\']} label mismatches. Process-wall runtime was {run[\'process_wall_seconds\']:.2f} seconds. Peak PyTorch allocated GPU memory was {run[\'metrics\'][\'peak_allocated_mib\']:.2f} MiB; this excludes other GPU allocations and is not total board memory.\\")\\n    table([[\'Condition\',\'AUC\',\'AUC drop from clean\']]+[[s[\'condition\'],f\\"{float(s[\'auc\']):.4f}\\",f\\"{float(s[\'auc_drop\']):.4f}\\"] for s in stats],[235,100,160])\\n    p(\'The clean subset AUC is 0.8817 with a 95% bootstrap interval of 0.8243-0.9311. Accuracy at threshold 0.5 is 87.5%. The largest observed AUC drop is 0.0110 under Gaussian noise. Its paired 95% interval extends to 0.0244, above 0.02. Therefore, a small observed drop does not establish a population robustness bound. Intervals use 2,000 stratified bootstrap resamples on the fixed subset.\')\\n    p(\'Earlier four-model runs had median T4 time 60.36 seconds and maximum observed corruption drop 0.0129. Those are different protocols and are not the final five-model measurements. Grad-CAM visualizations are qualitative attribution; they do not prove causal attention to manipulation artifacts, decision correctness, fairness, or generalization. Bonus points and official scores are determined by judges.\')\\n    p(\'Setup and submission\',\'Heading2\')\\n    p(\'The documented local setup targets Python 3.10. requirements.txt supplies compatible package pins; the original Colab freeze and original requirements are retained separately. Changing Python, numerical libraries or GPU runtime can change outputs, so run fresh verification in the target environment. New verification/validation runs write to outputs/reproduction/ rather than overwriting recorded evidence.\')\\n    p(\'Supply one Google Drive folder shared as Anyone with the link can view: source code or a public repository link with README, final predictions.csv, all five trained checkpoints with ensemble_config.json, inference_script.py, ensemble_worker_control.py, requirements.txt and this detailed report. The existing explanatory video is optional. The original participant package specifies a 5 GB model limit and 60-minute T4 inference limit; the scoring specification separately rewards faster inference.\')\\n    p(\'The repository audit checks file preservation, official IDs, source/weight hashes, setup metadata, CLI entry points and notebook consistency. It is not a new training run or independent T4 benchmark. Exact replay evidence describes the recorded GPU environment; the reorganized workflow still requires a fresh target-runtime replay before claiming new measurements.\')\\n    def footer(canvas,doc):\\n        canvas.setFont(\'Helvetica\',8);canvas.setFillColor(colors.HexColor(\'#5a6872\'))\\n        canvas.drawString(40,24,\'TEAM SMPS | Final recorded evidence | Official scores unknown\')\\n        canvas.drawRightString(A4[0]-40,24,f\'{doc.page} / 4\')\\n    OUT.parent.mkdir(parents=True,exist_ok=True)\\n    SimpleDocTemplate(str(OUT),pagesize=A4,leftMargin=40,rightMargin=40,topMargin=35,bottomMargin=40).build(story,onFirstPage=footer,onLaterPages=footer)\\n    print(OUT)\\n\\nif __name__==\'__main__\':main()\\n"}')
for name,source in scripts.items():
    Path(name).parent.mkdir(parents=True,exist_ok=True)
    Path(name).write_text(source,encoding="utf-8")
print("Installed final sources and supporting imports")
resources=json.loads('{"requirements.txt": "torch==2.14.1\\ntorchvision==0.29.1\\ntimm==1.0.31\\nnumpy==2.2.6\\npandas==2.2.3\\npillow==11.3.0\\nmatplotlib==3.10.0\\nscikit-learn==1.6.1\\nopencv-python-headless==5.0.0.93\\nreportlab==5.1.0\\npypdf==7.1.0\\npymupdf==1.28.2\\n", "requirements-training.txt": "-r requirements.txt\\n# Existing generic src/ pipeline uses the Albumentations 1.x API.\\nalbumentations==1.3.1\\ntqdm==4.68.3\\n", "README.md": "# SMPS | Deepfake Detection\\n\\n**IIIT Bangalore Deepfake ML Hackathon \\u00b7 Team SMPS**\\n\\nA five-model ensemble classifies aligned faces as **Real (0)** or **Fake (1)**. Fine-tuning uses the official dataset only. The final CSV contains 3,000 official test image IDs; confidence is the probability of **Fake**.\\n\\n## Project navigation\\n\\n| Deliverable | Location |\\n|---|---|\\n| Final test predictions | [predictions.csv](predictions.csv) |\\n| Official inference CLI | [inference_script.py](inference_script.py) and [worker controller](ensemble_worker_control.py) |\\n| Trained models and ensemble configuration | [weights/](weights/) |\\n| Official dataset and event instructions | [datasets/](datasets/) |\\n| Final technical report, four pages | [final_technical_report.pdf](docs/reports/final_technical_report.pdf) |\\n| Preserved earlier detailed report | [report_detailed.pdf](docs/reports/report_detailed.pdf) |\\n| Concise technical report, two pages | [report.pdf](docs/reports/report.pdf) |\\n| Final measured status | [FINAL_STATUS.md](docs/FINAL_STATUS.md) |\\n| Offline results dashboard | [results.html](presentation/results.html) |\\n| Existing explanatory video | [Explanatory Video.mp4](presentation/Explanatory%20Video.mp4) |\\n| Submission requirements | [Submission guide](docs/SUBMISSION.md) |\\n\\nUse the final technical report for submission. Earlier reports are preserved as history and contain superseded performance/score claims. The final report and final status distinguish the measured five-model results from earlier runs. See the [setup audit](docs/SETUP_AUDIT.md) for verification scope and remaining target-runtime checks.\\n\\n## Repository structure\\n\\n```text\\nSMPS/\\n|-- inference_script.py           Required hackathon CLI\\n|-- ensemble_worker_control.py    Sequential ensemble workers\\n|-- config.py                     Training and dataset configuration\\n|-- requirements.txt              Recorded final runtime dependencies\\n|-- predictions.csv               Final verified test predictions\\n|-- weights/                      Checkpoints, ensemble config and exports\\n|-- datasets/                     Official archive and participant instructions\\n|-- data/                         Saved train/validation splits\\n|-- src/                          Models, datasets, augmentation and training\\n|-- scripts/                      Preparation, evaluation and reporting tools\\n|-- notebooks/\\n|   |-- SMPS_Final_Five_Model_Verification.ipynb\\n|   |-- SMPS_T4_Efficiency_Benchmark.ipynb\\n|   `-- training/                 All original training/result notebooks\\n|-- outputs/\\n|   |-- final_verification/       Final FP32 replay and timing evidence\\n|   |-- final_validation/         Five-model robustness statistics\\n|   |-- models/                   Results grouped by architecture\\n|   |-- predictions/              Individual-model and earlier predictions\\n|   |-- gradcam/                  Explainability visualizations\\n|   `-- ...                       Preserved earlier runs, figures and logs\\n|-- docs/                         Setup, protocols, reports and handover\\n`-- presentation/                 Existing dashboard and video\\n```\\n\\n**Every original tracked file is retained.** The [reorganization manifest](docs/REORGANIZATION_MANIFEST.json) maps original paths to current paths and records original SHA-256 hashes. Historical evidence, final predictions, inference sources and binary assets retain their contents. The [previous README](docs/README_BEFORE_REORGANIZATION.md) is also preserved.\\n\\n## Reproduce inference\\n\\nRun from the repository root. Download actual model binaries with Git LFS; pointer files cannot be used for inference.\\n\\n```bash\\ngit lfs install\\ngit lfs pull --include=\\"weights/*,datasets/hackathon_dataset.zip\\"\\npython -m pip install --upgrade pip\\npython -m pip install -r requirements.txt\\npython inference_script.py --test_dir PATH/test/images --weights weights/ensemble_config.json --output fresh_predictions.csv\\n```\\n\\nThe CLI accepts the required `--test_dir`, `--weights`, and `--output` arguments. It defaults to FP32 and sequential model processes to limit memory use. Keep the worker controller beside the entry point and all five checkpoints beside the ensemble configuration.\\n\\nUse a new Python 3.10 virtual environment. The original measured Colab environment is recorded in [environment.txt](outputs/final_verification/environment.txt), and its original pins are preserved in [requirements_original_colab.txt](docs/environments/requirements_original_colab.txt). The current requirements use Python-3.10-compatible NumPy/PyMuPDF pins. This compatibility adjustment requires fresh verification in the target environment; exact recorded GPU results are not automatically transferable to a changed runtime. Earlier training notebooks retain their architecture-specific dependencies and hyperparameters.\\n\\n### Verify and evaluate\\n\\n```bash\\npython scripts/check_delivery.py\\npython scripts/verify_final.py --test-dir PATH/test/images\\npython scripts/validate_final.py --dataset-root PATH/hackathon_dataset\\npython scripts/build_results.py\\n```\\n\\nFresh evaluation writes to `outputs/reproduction/verification/` and `outputs/reproduction/validation/`. Choose a new `--output-dir` for another run; existing completed runs are protected. `build_results.py` rebuilds the dashboard from the recorded final evidence, not fresh reproduction folders. Open `presentation/results.html` in a browser. CPU runs do not establish T4 timing.\\n\\nFor Colab, upload [the final verification notebook](notebooks/SMPS_Final_Five_Model_Verification.ipynb), select a Tesla T4 runtime and run its cells. It downloads official assets, generates a fresh candidate, independently replays inference and measures robustness. Its results download excludes large weights and datasets.\\n\\n## Dataset and training\\n\\nExtract `datasets/hackathon_dataset.zip` into `datasets/extracted/`. The default root is `datasets/extracted/hackathon_dataset`, containing `train_metadata.csv`, `train/real`, `train/fake`, and `test/images`. For another extraction layout, set the directory containing the metadata explicitly:\\n\\n```powershell\\n$env:SMPS_DATASET_ROOT = \\"D:\\\\path\\\\to\\\\hackathon_dataset\\"\\npython -m pip install -r requirements-training.txt\\npython scripts/create_split.py\\npython scripts/eda.py\\npython -m src.train\\n```\\n\\nThe generic training pipeline in `src/` is a baseline scaffold, not a one-command reproduction of every final architecture. Architecture-specific hyperparameters, training outputs and challenges remain in [training notebooks](notebooks/training/) and [model results](outputs/models/). Saved splits contain 5,600 training and 1,400 validation images. Test images must not be used for training or tuning. Training dependencies add the Albumentations 1.x API used by this scaffold.\\n\\n## Final ensemble\\n\\n| Model | Fixed blend weight | Recorded individual peak validation AUC |\\n|---|---:|---:|\\n| EfficientNetV2-S | 0.40 | 0.9025 |\\n| ConvNeXt-Tiny | 0.25 | 0.8973 |\\n| Swin-Tiny | 0.15 | 0.8953 |\\n| MaxViT-Tiny | 0.12 | 0.8902 |\\n| DenseNet-121 | 0.08 | 0.8905 |\\n\\nEach model averages original and horizontal-flip probabilities before weighted blending. Blend weights are heuristic. Individual peaks and final validation use different samples, so these numbers do not establish an ensemble gain over the best individual model.\\n\\n## Measured final results\\n\\n| Check | Recorded result |\\n|---|---|\\n| Independent FP32 replay | All 3,000 labels and probabilities matched exactly |\\n| Tesla T4 process-wall time | 172.34 seconds |\\n| Peak PyTorch allocated GPU memory | 2,796.93 MiB; excludes other GPU allocations |\\n| Clean AUC on 200 balanced validation images | 0.8817; 95% bootstrap interval 0.8243\\u20130.9311 |\\n| Accuracy on that subset at threshold 0.5 | 87.5% |\\n| Largest observed corruption AUC drop | 0.0110 under Gaussian noise |\\n\\nSee [final replay evidence](outputs/final_verification/verification.json), [validation statistics](outputs/final_validation/statistics.csv), and [final status](docs/FINAL_STATUS.md). Noise-drop uncertainty extends above 0.02. Private test labels, official scores and population robustness guarantees remain unknown. Grad-CAM is qualitative attribution rather than proof of correct reasoning.\\n\\n## Hackathon submission\\n\\nSupply one Google Drive folder shared as **Anyone with the link can view**, containing source or a public repository link, trained weights, official test predictions and detailed documentation. The event requests an optional 3\\u20135 minute explanatory video; the existing video is preserved and its duration has not been revalidated. See the [submission guide](docs/SUBMISSION.md) for file locations and original event instructions.\\n\\nEvaluation weights: test performance **50%**, efficiency **20%**, robustness **20%**, documentation **10%**. Fine-tune exclusively on the provided dataset, use original and reproducible code, and provide only one final team submission; only the latest upload is evaluated.\\n", "config.py": "\\"\\"\\"\\nconfig.py\\n=========\\nCentral configuration for the Deepfake Detection project.\\nEdit the DATASET_ROOT to point to your participant package dataset.\\nAll other scripts import from here \\u2014 no hardcoded paths elsewhere.\\n\\"\\"\\"\\n\\nimport os\\nimport torch\\nfrom pathlib import Path\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# PATHS \\u2014 Edit DATASET_ROOT to match your machine\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nPROJECT_ROOT = Path(__file__).resolve().parent\\nDATASET_ROOT = Path(os.environ.get(\\"SMPS_DATASET_ROOT\\", str(PROJECT_ROOT / \\"datasets\\" / \\"extracted\\" / \\"hackathon_dataset\\")))\\n\\nTRAIN_REAL_DIR  = DATASET_ROOT / \\"train\\" / \\"real\\"\\nTRAIN_FAKE_DIR  = DATASET_ROOT / \\"train\\" / \\"fake\\"\\nTEST_DIR        = DATASET_ROOT / \\"test\\" / \\"images\\"\\nTRAIN_META_CSV  = DATASET_ROOT / \\"train_metadata.csv\\"\\nSAMPLE_SUB_CSV  = DATASET_ROOT / \\"sample_submission.csv\\"\\n\\nDATA_DIR        = PROJECT_ROOT / \\"data\\"\\nWEIGHTS_DIR     = PROJECT_ROOT / \\"weights\\"\\nOUTPUTS_DIR     = PROJECT_ROOT / \\"outputs\\"\\n\\nTRAIN_SPLIT_CSV = DATA_DIR / \\"train_split.csv\\"\\nVAL_SPLIT_CSV   = DATA_DIR / \\"val_split.csv\\"\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# MODEL SETTINGS\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nIMAGE_SIZE   = 224          # Must match training if you change inference script\\nBATCH_SIZE   = 32           # Training batch size\\nINFER_BATCH  = 64           # Inference batch size (larger = faster)\\nNUM_WORKERS  = 4\\nTHRESHOLD    = 0.5          # Fake probability threshold for binary label\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# TRAINING SETTINGS\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nEPOCHS         = 25\\nLR             = 1e-4\\nWEIGHT_DECAY   = 1e-4\\nSEED           = 42\\nVAL_SPLIT      = 0.20       # 80/20 split\\nBACKBONE       = \\"efficientnet_b4\\"   # Options: efficientnet_b4, efficientnet_b2, convnext_tiny\\n\\n# ImageNet normalisation stats (used for pre-trained backbones)\\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\\nIMAGENET_STD  = [0.229, 0.224, 0.225]\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# MIXED PRECISION\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# AMP only works on CUDA \\u2014 auto-disabled on CPU to suppress warnings\\nUSE_AMP = torch.cuda.is_available()\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# BEST WEIGHTS FILE (used by inference_script.py)\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nBEST_WEIGHTS = WEIGHTS_DIR / \\"best_model.pth\\"\\n\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n# DEVICE \\u2014 resolved once here so all scripts can import it\\n# \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\nDEVICE = \\"cuda\\" if torch.cuda.is_available() else \\"cpu\\"\\n"}')
for name,source in resources.items():
    Path(name).write_text(source,encoding="utf-8")


In [ ]:
import zipfile, hashlib
extract=project/'datasets/extracted'
extract.mkdir(exist_ok=True)
with zipfile.ZipFile('datasets/hackathon_dataset.zip') as z:
    for item in z.infolist():
        target=(extract/item.filename).resolve()
        assert target.is_relative_to(extract.resolve())
        if not item.is_dir():
            target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(z.read(item))
metadata=list(extract.rglob('train_metadata.csv'))
assert len(metadata)==1, 'Expected one official dataset'
dataset=metadata[0].parent
print('Dataset:',dataset)


In [ ]:
# Generate a fresh FP32 candidate, preserve the original, and replay independently.
import json,shutil,subprocess
original=Path('outputs/predictions/predictions_original.csv')
if not original.exists():shutil.copy2('predictions.csv',original)
def stream_run(command):
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in process.stdout:print(line,end='',flush=True)
    return process.wait()
folder=Path('outputs/reproduction/verification');folder.mkdir(parents=True,exist_ok=True)
candidate=folder/'submission_candidate.csv'
assert stream_run(['python','-u','inference_script.py','--test_dir',str(dataset/'test/images'),'--weights','weights/ensemble_config.json','--output',str(candidate),'--metrics_json',str(folder/'candidate_metrics.json')])==0
shutil.copy2(candidate,'predictions.csv')
assert stream_run(['python','-u','scripts/verify_final.py','--test-dir',str(dataset/'test/images'),'--runs','1','--output-dir',str(folder)])==0,'Independent reproduction failed; inspect diagnostics.'
print((folder/'verification.json').read_text())


In [ ]:
# Measure validation afresh; preserve the historical final evidence separately.
import pandas as pd, html, json
assert stream_run(['python','-u','scripts/validate_final.py','--dataset-root',str(dataset),'--output-dir','outputs/reproduction/validation']) == 0
with open('outputs/reproduction/verification/environment.txt','w') as env_log:
    subprocess.run(['python','-m','pip','freeze'],stdout=env_log,check=True)
from importlib.metadata import version
packages=['torch','torchvision','timm','numpy','pandas','pillow','matplotlib','scikit-learn','opencv-python-headless','reportlab','pypdf','pymupdf']
Path('outputs/reproduction/verification/requirements_runtime.txt').write_text(''.join(name+'=='+version(name).split('+')[0]+'\n' for name in packages))
# The dashboard remains the recorded-run presentation.
# Generate the final report from the preserved recorded evidence.
assert stream_run(['python','scripts/generate_final_report.py']) == 0
from pypdf import PdfReader
assert len(PdfReader('docs/reports/final_technical_report.pdf').pages)>=4
print('FINAL FIVE-MODEL VALIDATION (200 held-out images):')
display(pd.read_csv('outputs/reproduction/validation/statistics.csv'))
print('FINAL REPRODUCTION RESULT:')
print(Path('outputs/reproduction/verification/verification.json').read_text())
from IPython.display import display, HTML
page = html.escape(Path('presentation/results.html').read_text(), quote=True)
display(HTML('<iframe style="width:100%;height:900px;border:0" allowfullscreen srcdoc="'+page+'"></iframe>'))


In [ ]:
import zipfile,json
from google.colab import files
verification=json.loads(Path('outputs/reproduction/verification/verification.json').read_text())
assert verification['weights_to_csv_status']=='passed'
archive='/content/SMPS_verified_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['presentation/results.html','docs/reports/final_technical_report.pdf','predictions.csv','outputs/predictions/predictions_original.csv','requirements.txt','weights/ensemble_config.json']:
        z.write(name)
    for name in scripts:z.write(name)
    for folder in ['outputs/reproduction/verification','outputs/reproduction/validation']:
        for path in Path(folder).rglob('*'):
            if path.is_file():z.write(path)
print('Independent reproduction passed:',verification['runs'][0]['reproduction'])
files.download(archive)
